In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType
from pyspark.sql import DataFrame, SparkSession, Window, Column
from pyspark.sql.functions import (
    col, trim, upper, current_timestamp, lit, when, row_number, 
    to_date, array, concat_ws, array_compact, count
)
import uuid
from datetime import UTC, datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql.types import DecimalType, IntegerType, StringType
from decimal import Decimal
from functools import reduce
from operator import add, or_
from dataclasses import dataclass
from pyspark.sql.types import StructType



In [0]:
# Pipeline configuration

CATALOG_NAME = "data2026acc123456"
SCHEMA_NAME = "team3"

BASE_SOURCE_PATH = (
    f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/test_datasets"
)

BASE_CHECKPOINT_PATH = (
    f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/test_checkpoints"
)

AUDIT_LOG_TABLE = (
    f"{CATALOG_NAME}.{SCHEMA_NAME}.test_bronze_ingestion_audit"
)

spark.conf.set("spark.sql.session.timeZone", "UTC")


In [0]:
# Helper: Safe table existence check for Unity Catalog 3-level namespace

def check_table_exists(session, full_table_name: str) -> bool:
    """
    Safely verifies if a 3-level namespace table exists in Unity Catalog.
    Returns True if table exists, False if it does not.
    """
    try:
        return session.catalog.tableExists(full_table_name)
    except Exception:
        return False


# Audit logging

def log_audit_metrics(
    session,
    audit_table: str,
    target_table: str,
    batch_id: int,
    input_count: int,
    written_count: int,
    rejected_count: int,
    duration_sec: float,
    status: str,
    error_message: str = None
):
    """Write batch-level metrics to the audit Delta table."""

    audit_data = [(
        target_table,
        int(batch_id),
        int(input_count),
        int(written_count),
        int(rejected_count),
        float(duration_sec),
        status,
        str(error_message)[:4000] if error_message else None
    )]

    schema = """
        target_table STRING,
        batch_id LONG,
        input_count LONG,
        written_count LONG,
        rejected_count LONG,
        duration_sec DOUBLE,
        status STRING,
        error_message STRING
    """

    audit_df = (
        session.createDataFrame(audit_data, schema=schema)
        .withColumn("logged_at", F.current_timestamp())
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )

# Micro-batch processing

def process_micro_batch(
    batch_df,
    batch_id: int,
    target_table: str,
    audit_table: str
):
    """Enrich and write a micro-batch, then record ingestion metrics."""

    session = batch_df.sparkSession
    start_time = time.time()

    input_count = 0
    written_count = 0

    try:
        input_count = batch_df.count()

        if input_count == 0:
            return

        # Ensure the rescued-data column exists.
        if "_rescued_data" not in batch_df.columns:
            batch_df = batch_df.withColumn(
                "_rescued_data",
                F.lit(None).cast("string")
            )

        # Include source fields and rescued content in the hash.
        # Exclude the helper column containing the physical file path.
        payload_cols = sorted([
            name for name in batch_df.columns
            if name != "_bronze_file_path"
        ])

        # JSON serialization supports nested structs and arrays.
        # Compute the hash before adding ingestion metadata.
        enriched_df = batch_df.withColumn(
            "record_hash",
            F.sha2(
                F.to_json(
                    F.struct(*[
                        F.col(f"`{name}`")
                        for name in payload_cols
                    ]),
                    options={"ignoreNullFields": "false"}
                ),
                256
            )
        )

        # Preserve the batch identifier supplied by the source CSV.
        if "batch_id" in enriched_df.columns:
            enriched_df = enriched_df.withColumnRenamed(
                "batch_id",
                "source_batch_id"
            )

        enriched_df = (
            enriched_df
            .withColumn(
                "ingestion_timestamp",
                F.current_timestamp()
            )
            .withColumn(
                "source_filename",
                F.col("_bronze_file_path")
            )
            .withColumn(
                "batch_id",
                F.lit(batch_id).cast("long")
            )
            .withColumn(
                "load_status",
                F.when(
                    F.col("_rescued_data").isNotNull(),
                    F.lit("RESCUED")
                ).otherwise(F.lit("SUCCESS"))
            )
            .drop("_bronze_file_path")
        )

        rescued_count = enriched_df.filter(
            F.col("load_status") == "RESCUED"
        ).count()

        # ----------------------------------------------------------------------
        # CHECK IF TABLE EXISTS: CREATE FROM SCRATCH vs APPEND / MERGE
        # ----------------------------------------------------------------------
        table_exists = check_table_exists(session, target_table)

        if not table_exists:
            print(f"Table {target_table} does not exist. Creating from scratch...")
            (
                enriched_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(target_table)
            )
            written_count = input_count

        else:
            print(f"Table {target_table} exists. Merging / appending new records...")
            enriched_df.createOrReplaceTempView("staged_micro_batch")

            session.sql(f"""
                MERGE INTO {target_table} AS target
                USING staged_micro_batch AS source
                ON target.record_hash = source.record_hash
                WHEN NOT MATCHED THEN INSERT *
            """)

            # Read the actual number of inserted rows.
            last_commit = (
                session.sql(f"DESCRIBE HISTORY {target_table}")
                .orderBy(F.desc("version"))
                .select("operationMetrics")
                .first()
            )

            metrics = last_commit["operationMetrics"] or {}
            written_count = int(metrics.get("numTargetRowsInserted", 0))

        duration_sec = round(time.time() - start_time, 2)
        rejected_count = 0

        log_audit_metrics(
            session=session,
            audit_table=audit_table,
            target_table=target_table,
            batch_id=batch_id,
            input_count=input_count,
            written_count=written_count,
            rejected_count=rejected_count,
            duration_sec=duration_sec,
            status=(
                "SUCCESS_WITH_RESCUED"
                if rescued_count > 0
                else "SUCCESS"
            )
        )

        print(
            f"[{target_table}] Batch {batch_id}: "
            f"input={input_count}, "
            f"inserted={written_count}, "
            f"skipped={input_count - written_count}, "
            f"rescued_in_input={rescued_count}, "
            f"duration={duration_sec}s"
        )

    except Exception as err:
        duration_sec = round(time.time() - start_time, 2)

        try:
            log_audit_metrics(
                session=session,
                audit_table=audit_table,
                target_table=target_table,
                batch_id=batch_id,
                input_count=input_count,
                written_count=written_count,
                rejected_count=0,
                duration_sec=duration_sec,
                status="FAILED",
                error_message=str(err)
            )
        except Exception as audit_err:
            print(f"Audit logging also failed: {audit_err}")

        raise


# ==============================================================================
# Ingestion function
# ==============================================================================

def ingest_to_bronze(
    source_path: str,
    file_format: str,
    target_table: str,
    checkpoint_path: str,
    audit_table: str
):
    """Load CSV files or JSON arrays into a Bronze Delta table."""

    file_format = file_format.lower()

    if file_format == "json":
        json_schema = (
            spark.read
            .option("multiLine", "true")
            .option("primitivesAsString", "true")
            .option("mode", "FAILFAST")
            .option("pathGlobFilter", "*.json")
            .json(source_path)
            .schema
        )

        required_columns = {
            "event_id",
            "equipment_id",
            "sensor",
            "reading"
        }

        missing_columns = (
            required_columns - set(json_schema.fieldNames())
        )

        if missing_columns:
            raise ValueError(
                f"Missing JSON fields: {sorted(missing_columns)}"
            )

        json_files = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "binaryFile")
            .option("pathGlobFilter", "*.json")
            .load(source_path)
        )

        raw_df = (
            json_files
            .select(
                F.col("path").alias("_bronze_file_path"),
                F.explode(
                    F.from_json(
                        F.decode(F.col("content"), "UTF-8"),
                        ArrayType(json_schema),
                        options={"mode": "FAILFAST"}
                    )
                ).alias("event")
            )
            .select(
                "event.*",
                "_bronze_file_path"
            )
            .withColumn(
                "_rescued_data",
                F.lit(None).cast("string")
            )
        )

    else:
        read_options = {
            "cloudFiles.format": file_format,
            "cloudFiles.schemaLocation": f"{checkpoint_path}/schema",
            "cloudFiles.schemaEvolutionMode": "addNewColumns",
            "rescuedDataColumn": "_rescued_data",
            "cloudFiles.inferColumnTypes": "true"
        }

        if file_format == "csv":
            read_options["header"] = "true"

        raw_df = (
            spark.readStream
            .format("cloudFiles")
            .options(**read_options)
            .load(source_path)
            .select(
                "*",
                F.col("_metadata.file_path").alias(
                    "_bronze_file_path"
                )
            )
        )

    query = (
        raw_df.writeStream
        .option(
            "checkpointLocation",
            f"{checkpoint_path}/data"
        )
        .trigger(availableNow=True)
        .foreachBatch(
            lambda df, b_id: process_micro_batch(
                df,
                b_id,
                target_table,
                audit_table
            )
        )
        .start()
    )

    query.awaitTermination()

    print(f"Ingestion completed for table: {target_table}\n")

# Run all four sources sequentially

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/equipment",
    file_format="csv",
    target_table=f"{CATALOG_NAME}.{SCHEMA_NAME}.test_bronze_equipment",
    checkpoint_path=f"{BASE_CHECKPOINT_PATH}/equipment",
    audit_table=AUDIT_LOG_TABLE
)

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/maintenance_history",
    file_format="csv",
    target_table=(
        f"{CATALOG_NAME}.{SCHEMA_NAME}.test_bronze_maintenance_history"
    ),
    checkpoint_path=f"{BASE_CHECKPOINT_PATH}/maintenance_history",
    audit_table=AUDIT_LOG_TABLE
)

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/work_orders",
    file_format="csv",
    target_table=f"{CATALOG_NAME}.{SCHEMA_NAME}.test_bronze_work_orders",
    checkpoint_path=f"{BASE_CHECKPOINT_PATH}/work_orders",
    audit_table=AUDIT_LOG_TABLE
)

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/sensor_events",
    file_format="json",
    target_table=f"{CATALOG_NAME}.{SCHEMA_NAME}.test_bronze_sensor_events",
    checkpoint_path=(
        f"{BASE_CHECKPOINT_PATH}/sensor_events_binary_v1"
    ),
    audit_table=AUDIT_LOG_TABLE
)

print("All Bronze ingestions finished and audited.")

Ingestion completed for table: data2026acc123456.team3.test_bronze_equipment

Ingestion completed for table: data2026acc123456.team3.test_bronze_maintenance_history

Ingestion completed for table: data2026acc123456.team3.test_bronze_work_orders

Ingestion completed for table: data2026acc123456.team3.test_bronze_sensor_events

All Bronze ingestions finished and audited.


## Silver Equipment Ingestion

In [0]:
MAX_POWER_KW = 5000

BRONZE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.test_bronze_equipment"
SILVER_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.test_silver_equipment"
QUARANTINE_LOG = f"{CATALOG_NAME}.{SCHEMA_NAME}.test_quarantine_log"
DQ_LOG = f"{CATALOG_NAME}.{SCHEMA_NAME}.test_dq_results"

def check_contracts(df: DataFrame) -> DataFrame:
    # Step 0: Standard processing applied
    return df

def standardise(df: DataFrame) -> DataFrame:
    # Step 1: Standard processing applied
    return (df
        .withColumn("equipment_id", when(trim(col("equipment_id")) == "", lit(None)).otherwise(trim(col("equipment_id"))))
        .withColumn("install_date", to_date(col("install_date")))
        .withColumn("decommission_date", to_date(col("decommission_date")))
        .withColumn("country", upper(trim(col("country"))))
        .withColumn("site_code", upper(trim(col("site_code"))))
        .withColumn("criticality", upper(trim(col("criticality"))))
    )

def flag_duplicates(df: DataFrame) -> DataFrame:
    # Step 2 & 3: Standard processing applied
    window_spec = Window.partitionBy("equipment_id").orderBy(col("ingestion_timestamp").desc())
    return (df
        .withColumn("rn", row_number().over(window_spec))
        .withColumn("is_old_version", col("rn") > 1)
        .drop("rn")
    )

def check_equipment_dates(df: DataFrame) -> DataFrame:
    # Step 4: Standard processing applied
    return df.withColumn("flag_invalid_date", col("decommission_date") < col("install_date"))

def check_power_limits(df: DataFrame, max_power: int) -> DataFrame:
    # Step 5: Standard processing applied
    return df.withColumn("flag_invalid_power", (col("rated_power_kw") < 0) | (col("rated_power_kw") > max_power))

def check_vibration_limits(df: DataFrame) -> DataFrame:
    # Step 6: Standard processing applied
    return df.withColumn("flag_invalid_vibration", col("rated_vibration_max_mm_s") < 0)

def check_missing_keys(df: DataFrame) -> DataFrame:
    # Step 7: Standard processing applied
    return df.withColumn("flag_missing_id", col("equipment_id").isNull())

def check_rules(df: DataFrame) -> DataFrame:
    # Step 8: Combine flags and quarantine reasoning
    df_rules = df.withColumn(
        "quarantine_reasons_array",
        array(
            when(col("is_old_version"), lit("EQ_OLD_VERSION")),
            when(col("flag_invalid_date"), lit("EQ_DATE_LOGIC")),
            when(col("flag_invalid_power"), lit("EQ_POWER_BOUNDS")),
            when(col("flag_invalid_vibration"), lit("EQ_NEG_VIBRATION")),
            when(col("flag_missing_id"), lit("EQ_MISSING_ID"))
        )
    )
    
    return (df_rules
        .withColumn("quarantine_reason", concat_ws(" | ", array_compact(col("quarantine_reasons_array"))))
        .withColumn("is_quarantined", col("quarantine_reason") != "")
    )

def build_silver(df: DataFrame) -> tuple[DataFrame, DataFrame, DataFrame]:
    # Step 9: Split streams and aggregate metrics
    drop_cols = [
        "is_old_version", "flag_invalid_date", "flag_invalid_power", 
        "flag_invalid_vibration", "flag_missing_id", "quarantine_reasons_array", 
        "quarantine_reason", "is_quarantined"
    ]
    
    df_valid = df.filter(~col("is_quarantined")).drop(*drop_cols)
    
    df_quarantine = (df
        .filter(col("is_quarantined"))
        .withColumn("table_source", lit("equipment"))
        .withColumn("quarantine_timestamp", current_timestamp())
    )
    
    df_dq = (df
        .select(
            count("*").alias("total_records_processed"),
            count(when(~col("is_quarantined"), True)).alias("valid_records"),
            count(when(col("is_quarantined"), True)).alias("quarantined_records")
        )
        .withColumn("pipeline_run_ts", current_timestamp())
        .withColumn("dataset_name", lit("equipment"))
    )
    
    return df_valid, df_quarantine, df_dq


# Step 10: Orchestration and Delta Operations
spark = spark or SparkSession.getActiveSession()
if spark is None:
    raise RuntimeError("Active SparkSession required.")

df_bronze = spark.read.table(BRONZE_TABLE)

df = check_contracts(df_bronze)
df = standardise(df)
df = flag_duplicates(df)
df = check_equipment_dates(df)
df = check_power_limits(df, MAX_POWER_KW)
df = check_vibration_limits(df)
df = check_missing_keys(df)
df = check_rules(df)

df_valid, df_quarantine, df_dq = build_silver(df)

# Check if table exists BEFORE creating the temporary view
silver_table_exists = spark.catalog.tableExists(SILVER_TABLE)

if not silver_table_exists:
    # Table does not exist: create from scratch with schema and initial records
    df_valid.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(SILVER_TABLE)
    print(f"Created table {SILVER_TABLE} from scratch.")
else:
    # Table exists: create view and perform idempotent merge
    df_valid.createOrReplaceTempView("test_staged_silver_equipment")
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING test_staged_silver_equipment AS source
        ON target.equipment_id = source.equipment_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table {SILVER_TABLE}.")

df_quarantine.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(QUARANTINE_LOG)
df_dq.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(DQ_LOG)

print("Silver Equipment Table Updated")

Merged into existing table data2026acc123456.team3.test_silver_equipment.
Silver Equipment Table Updated


## Silver Maintenance History Ingestion

In [0]:
# 1. Config & Setup
CONFIG = {
    "bronze_loaded_status": "SUCCESS",
    "bronze_column_map": {
        "batch_id": "_batch_id",
        "source_batch_id": "batch_id",
        "ingestion_timestamp": "_ingestion_ts",
        "source_filename": "_source_file",
        "record_hash": "_record_hash",
        "load_status": "_load_status",
    },
    "timestamp_pattern": "yyyy-MM-dd'T'HH:mm:ss'Z'",
    "date_pattern": "yyyy-MM-dd",
    "legacy_source_system": "LEGACY_CMMS",
    "no_parts_activity_types": ["Inspect", "Align", "Adjust", "Calibrate"],
    "labour_sentinels": [999],
    "min_booking_h": 0.25,
    "allowed_factors": [1, 2],
    "factor_tolerance": 0.05,
    "max_activities_per_work_order": 2,
    "codes": {
        "activity_type": [
            "Inspect",
            "Replace",
            "Align",
            "Clean",
            "Service",
            "Calibrate",
            "Lubricate",
            "Repair",
            "Adjust",
        ],
        "failure_mode_code": [
            "NONE",
            "BRG-WEAR",
            "MISALIGN",
            "IMBAL",
            "OVERHEAT",
            "LUB-FAIL",
            "WIND-INS",
            "SEAL-LEAK",
            "CAVIT",
            "VALVE-FAIL",
            "GEAR-WEAR",
            "CLOG",
        ],
        "root_cause_code": [
            "RC-WEAR",
            "RC-CONTAM",
            "RC-LUB",
            "RC-INST",
            "RC-OPER",
            "RC-MAINT",
            "RC-DESIGN",
            "RC-MATL",
        ],
        "detection_method": [
            "scheduled_pm",
            "inspection",
            "operator_report",
            "condition_monitoring",
            "alarm",
        ],
        "outcome": ["Resolved", "Temporary fix"],
    },
    "technician_pattern": "^TECH-[0-9]{2}$",
    "component_to_code": {
        "Whole asset": "GENERAL",
        "Shaft coupling": "COUPLING",
        "Drive-end bearing": "BRG-DE",
        "Bearing and coupling": "BRG-DE",
        "Non-drive-end bearing": "BRG-NDE",
        "Cooling system / heat exchanger": "COOLING-SYS",
        "Lubrication system": "LUBE-SYS",
        "Sensor package": "SENSOR-PKG",
        "Oil filter": "OIL-FILTER",
        "Rotor assembly": "ROTOR",
        "Stator winding": "STATOR-WDG",
        "Mechanical seal": "SEAL-MECH",
        "Impeller": "IMPELLER",
        "Valve assembly": "VALVE",
        "Gear set": "GEAR-SET",
        "Suction strainer / filter": "STRAINER",
        "Fan blade set": "FAN-BLADE",
    },
    "tables": {
        "silver": "test_silver_maintenance_history",
        "quarantine": "test_silver_dq_quarantine",
        "audit": "test_dq_results",
    },
    "rule_meta": {
        "TYPE_CAST": ("Error", "Validity", "Non-empty value that doesn't parse"),
        "DUP_EXACT": ("Drop", "Uniqueness", "Exact duplicate of business columns"),
        "VERSION_LATEST": ("Drop", "Uniqueness", "Superseded version"),
        "SOURCE_KEY_FORMAT": ("Error", "Validity", "source_system_key format"),
        "DOMAIN_CODES": ("Error", "Validity", "Codes in their lists"),
        "COMPONENT_CODE_MAP": ("Error", "Validity", "component to component_code"),
        "WO_NULL_NON_LEGACY": ("Error", "Completeness", "Null work_order_id on CMMS"),
        "RC_NULL_IFF_NONE": ("Error", "Validity", "Root cause null iff NONE"),
        "NDD_IFF_PLANNED": ("Error", "Validity", "Next due date set iff NONE"),
        "EQ_MISSING": ("Row quarantine", "Completeness", "equipment_id null"),
        "TECH_ID_MISSING": ("Field quarantine", "Completeness", "technician_id null"),
        "ACTIVITIES_PER_WO": ("Warn", "Consistency", "More than 2 activities on WO"),
        "TS_END_BEFORE_START": ("Field quarantine", "Validity", "End before start"),
        "TS_AFTER_EXTRACT": ("Warn", "Validity", "Start or end after ingestion_ts"),
        "TS_MODIFIED_BEFORE_END": ("Warn", "Validity", "last_modified_ts < end"),
        "NDD_AFTER_END": ("Warn", "Validity", "Next due date not after end date"),
        "C1": ("Fix", "Validity", "Cost zeroed"),
        "C2": ("Field quarantine", "Validity", "Negative cost nulled"),
        "Q1": ("Fix", "Validity", "Quantity zeroed"),
        "Q2": ("Fix", "Validity", "Quantity made positive"),
        "PARTS_QTY_COST_CONSISTENT": ("Error", "Consistency", "Quantity 0 iff cost 0"),
        "L1": ("Field quarantine", "Validity", "Sentinel 999 nulled"),
        "L2": ("Fix", "Validity", "Minutes converted to hours"),
        "L3": ("Fix", "Validity", "Negative hours made positive"),
        "LABOR_UNEXPLAINED_RATIO": (
            "Error",
            "Validity",
            "Factor not 1 or 2 after fixes",
        ),
    },
}

SOURCE_COLS = [
    "maintenance_id",
    "work_order_id",
    "equipment_id",
    "activity_start_ts",
    "activity_end_ts",
    "activity_type",
    "component",
    "component_code",
    "failure_mode_code",
    "root_cause_code",
    "detection_method",
    "action_description",
    "parts_qty",
    "parts_cost_eur",
    "labor_hours",
    "technician_id",
    "outcome",
    "next_due_date",
    "last_modified_ts",
    "record_version",
    "source_file",
    "ingestion_ts",
    "batch_id",
    "source_system",
    "source_system_key",
]

LINEAGE_COLS = [
    "source_file",
    "ingestion_ts",
    "batch_id",
    "source_system",
    "source_system_key",
]
BUSINESS_COLS = [c for c in SOURCE_COLS if c not in LINEAGE_COLS]

B = lambda e: F.coalesce(e, F.lit(False))
FLAG = lambda n: f"_f_{n}"

# 2. Core Functions


def clean_maintenance_history(bronze_df: DataFrame, config: dict = CONFIG) -> DataFrame:
    """
    Cleans and standardizes the maintenance history bronze data[cite: 1].
    """

    def _prepare_and_standardise(df: DataFrame) -> DataFrame:
        m = config["bronze_column_map"]
        df = df.select(
            *[
                F.col(f"`{c}`").alias(m.get(c, c))
                for c in df.columns
                if c != "_rescued_data"
            ]
        )
        df = df.withColumn("_source_system", F.col("source_system"))
        df = df.filter(F.col("_load_status") == config["bronze_loaded_status"])

        # Extract JSON for quarantine
        df = df.withColumn(
            "_w_source_json",
            F.to_json(F.struct(*[c for c in df.columns if not c.startswith("_")])),
        )

        # Standardise: Trim strings
        for c, t in df.dtypes:
            if c in SOURCE_COLS and t == "string":
                df = df.withColumn(
                    c, F.when(F.trim(c) == "", None).otherwise(F.trim(c))
                )

        # Track TYPE_CAST failures[cite: 1]
        df = df.withColumn(FLAG("TYPE_CAST"), F.lit(False))

        casts = [
            ("parts_cost_eur", "DECIMAL(12,2)", "try_cast"),
            ("labor_hours", "DECIMAL(8,2)", "try_cast"),
            ("parts_qty", "INT", "try_cast"),
            ("record_version", "INT", "try_cast"),
            ("activity_start_ts", config["timestamp_pattern"], "try_to_timestamp"),
            ("activity_end_ts", config["timestamp_pattern"], "try_to_timestamp"),
            ("last_modified_ts", config["timestamp_pattern"], "try_to_timestamp"),
            ("ingestion_ts", config["timestamp_pattern"], "try_to_timestamp"),
            ("next_due_date", config["date_pattern"], "try_to_date"),
        ]

        for c, fmt, fn in casts:
            fmt = fmt.replace(
                "'", "\\'"
            )  # pattern's 'T'/'Z' quotes break the SQL literal
            expr_str = (
                f"{fn}({c}, '{fmt}')" if fn != "try_cast" else f"{fn}({c} AS {fmt})"
            )
            # Fails TYPE_CAST if originally not null/empty, but casting results in null[cite: 1]
            df = df.withColumn(
                FLAG("TYPE_CAST"),
                F.col(FLAG("TYPE_CAST"))
                | (
                    F.col(c).isNotNull()
                    & (F.trim(F.col(c).cast("string")) != "")
                    & F.expr(expr_str).isNull()
                ),
            )
            df = df.withColumn(c, F.expr(expr_str))

        return df

    def _deduplicate(df: DataFrame) -> DataFrame:
        bhash = F.sha2(
            F.to_json(
                F.struct(*[F.col(c).cast("string") for c in BUSINESS_COLS]),
                {"ignoreNullFields": "false"},
            ),
            256,
        )
        w_exact = Window.partitionBy("_bhash").orderBy(
            "_ingestion_ts", "_source_file", "_record_hash"
        )
        df = df.withColumn("_bhash", bhash).withColumn(
            FLAG("DUP_EXACT"), F.row_number().over(w_exact) > 1
        )

        w_version = Window.partitionBy("maintenance_id", FLAG("DUP_EXACT")).orderBy(
            F.desc_nulls_last("record_version"),
            F.desc_nulls_last("last_modified_ts"),
            F.desc_nulls_last("_ingestion_ts"),
            "_record_hash",
        )
        return df.withColumn(
            FLAG("VERSION_LATEST"),
            ~F.col(FLAG("DUP_EXACT"))
            & F.col("maintenance_id").isNotNull()
            & (F.row_number().over(w_version) > 1),
        )

    def _apply_validation_rules(df: DataFrame) -> DataFrame:
        df = df.withColumn(
            FLAG("SOURCE_KEY_FORMAT"),
            F.col("source_system_key")
            != F.concat_ws("|", F.col("source_system"), F.col("maintenance_id")),
        )

        map_expr = F.create_map(
            *[F.lit(x) for kv in config["component_to_code"].items() for x in kv]
        )
        df = df.withColumn(
            FLAG("COMPONENT_CODE_MAP"),
            F.col("component_code") != map_expr[F.col("component")],
        )

        domain_fail = (
            ~F.col("activity_type").isin(config["codes"]["activity_type"])
            | ~F.col("failure_mode_code").isin(config["codes"]["failure_mode_code"])
            | (
                F.col("root_cause_code").isNotNull()
                & ~F.col("root_cause_code").isin(config["codes"]["root_cause_code"])
            )
            | ~F.col("detection_method").isin(config["codes"]["detection_method"])
            | ~F.col("outcome").isin(config["codes"]["outcome"])
            | (
                F.col("technician_id").isNotNull()
                & ~F.col("technician_id").rlike(config["technician_pattern"])
            )
        )
        df = df.withColumn(FLAG("DOMAIN_CODES"), B(domain_fail))

        df = df.withColumn(
            FLAG("WO_NULL_NON_LEGACY"),
            F.col("work_order_id").isNull()
            & (F.col("source_system") != config["legacy_source_system"]),
        )
        df = df.withColumn(FLAG("EQ_MISSING"), F.col("equipment_id").isNull())
        df = df.withColumn(FLAG("TECH_ID_MISSING"), F.col("technician_id").isNull())
        df = df.withColumn(
            FLAG("RC_NULL_IFF_NONE"),
            F.col("root_cause_code").isNull() & (F.col("failure_mode_code") != "NONE"),
        )
        df = df.withColumn(
            FLAG("NDD_IFF_PLANNED"),
            F.col("next_due_date").isNotNull() & (F.col("failure_mode_code") != "NONE"),
        )

        w_wo = Window.partitionBy("work_order_id")
        df = df.withColumn(
            FLAG("ACTIVITIES_PER_WO"),
            F.col("work_order_id").isNotNull()
            & (
                F.count("maintenance_id").over(w_wo)
                > config["max_activities_per_work_order"]
            ),
        )
        df = df.withColumn(
            FLAG("TS_AFTER_EXTRACT"),
            B(
                (F.col("activity_start_ts") > F.col("ingestion_ts"))
                | (F.col("activity_end_ts") > F.col("ingestion_ts"))
            ),
        )
        df = df.withColumn(
            FLAG("TS_MODIFIED_BEFORE_END"),
            B(F.col("last_modified_ts") < F.col("activity_end_ts")),
        )
        df = df.withColumn(
            FLAG("NDD_AFTER_END"),
            B(
                F.col("next_due_date")
                <= F.coalesce(
                    F.col("activity_end_ts").cast("date"),
                    F.col("activity_start_ts").cast("date"),
                )
            ),
        )

        return df

    def _apply_fixes(df: DataFrame) -> DataFrame:
        df = df.withColumn(
            FLAG("TS_END_BEFORE_START"),
            B(F.col("activity_end_ts") < F.col("activity_start_ts")),
        )
        df = df.withColumn(
            "activity_end_ts",
            F.when(F.col(FLAG("TS_END_BEFORE_START")), None).otherwise(
                F.col("activity_end_ts")
            ),
        )

        c1 = F.col("activity_type").isin(config["no_parts_activity_types"]) & (
            F.col("parts_cost_eur") != 0
        )
        c2 = ~c1 & (F.col("parts_cost_eur") < 0)

        df = df.withColumn(FLAG("C1"), c1).withColumn(FLAG("C2"), c2)
        df = df.withColumn(
            "parts_cost_eur",
            F.when(c1, F.lit(0).cast(DecimalType(12, 2)))
            .when(c2, None)
            .otherwise(F.col("parts_cost_eur")),
        )
        df = df.withColumn(
            "parts_fix_rule",
            F.when(c1, "NO_PARTS_ACTIVITY_COST_ZEROED")
            .when(c2, "NEG_COST_NULLED")
            .otherwise("UNCHANGED"),
        )

        q1 = (F.col("parts_cost_eur") == 0) & (F.col("parts_qty") != 0)
        q2 = (F.col("parts_qty") < 0) & (
            (F.col("parts_cost_eur") != 0) | F.col("parts_cost_eur").isNull()
        )

        df = df.withColumn(FLAG("Q1"), q1).withColumn(FLAG("Q2"), q2)
        df = df.withColumn(
            "parts_qty",
            F.when(q1, F.lit(0).cast(IntegerType()))
            .when(q2, F.abs(F.col("parts_qty")))
            .otherwise(F.col("parts_qty")),
        )
        df = df.withColumn(
            "parts_fix_rule",
            F.when(
                q1,
                F.concat_ws(
                    ";", F.col("parts_fix_rule"), F.lit("ZERO_COST_QTY_ZEROED")
                ),
            )
            .when(
                q2,
                F.concat_ws(";", F.col("parts_fix_rule"), F.lit("NEG_QTY_CORRECTED")),
            )
            .otherwise(F.col("parts_fix_rule")),
        )
        df = df.withColumn(
            "parts_fix_rule", F.expr("replace(parts_fix_rule, 'UNCHANGED;', '')")
        )
        df = df.withColumn(
            FLAG("PARTS_QTY_COST_CONSISTENT"),
            (F.col("parts_cost_eur") == 0) & (F.col("parts_qty") != 0),
        )

        dur_h = F.expr(
            "(unix_timestamp(activity_end_ts) - unix_timestamp(activity_start_ts)) / 3600"
        )
        dur_floor = F.greatest(dur_h, F.lit(config["min_booking_h"]))
        ratio = F.abs(F.col("labor_hours")) / dur_floor
        factor_60 = F.round(ratio / 60)
        valid_factor_60 = factor_60.isin(*config["allowed_factors"]) & (
            F.abs((ratio / 60) - factor_60) <= config["factor_tolerance"]
        )

        l1 = F.col("labor_hours").isin(*config["labour_sentinels"])
        l2 = ~l1 & dur_h.isNotNull() & valid_factor_60
        l3 = ~l1 & ~l2 & (F.col("labor_hours") < 0)

        df = (
            df.withColumn(FLAG("L1"), B(l1))
            .withColumn(FLAG("L2"), B(l2))
            .withColumn(FLAG("L3"), B(l3))
        )
        df = df.withColumn(
            "labor_hours",
            F.when(l1, None)
            .when(l2, F.col("labor_hours") / 60)
            .when(l3, F.abs(F.col("labor_hours")))
            .otherwise(F.col("labor_hours")),
        ).withColumn("labor_hours", F.col("labor_hours").cast(DecimalType(8, 2)))

        df = df.withColumn(
            "labor_fix_rule",
            F.when(l1, "SENTINEL_999_NULLED")
            .when(l2, "MINUTES_TO_HOURS")
            .when(l3, "NEGATIVE_ABS")
            .otherwise("UNCHANGED"),
        )

        ratio_final = F.abs(F.col("labor_hours")) / dur_floor
        df = df.withColumn(
            FLAG("LABOR_UNEXPLAINED_RATIO"),
            F.col("labor_hours").isNotNull()
            & dur_h.isNotNull()
            & ~(
                F.round(ratio_final).isin(*config["allowed_factors"])
                & (
                    F.abs(ratio_final - F.round(ratio_final))
                    <= config["factor_tolerance"]
                )
            ),
        )

        return df

    return _apply_fixes(
        _apply_validation_rules(_deduplicate(_prepare_and_standardise(bronze_df)))
    )


def create_silver_tables(
    cleaned_df: DataFrame,
    catalog: str,
    schema: str,
    run_id: str,
    config: dict = CONFIG,
    spark=None,
) -> None:
    silver_table = f"{catalog}.{schema}.{config['tables']['silver']}"
    quar_table = f"{catalog}.{schema}.{config['tables']['quarantine']}"
    audit_table = f"{catalog}.{schema}.{config['tables']['audit']}"

    # Error Gate checks[cite: 1]
    error_rules = [
        FLAG("TYPE_CAST"),
        FLAG("SOURCE_KEY_FORMAT"),
        FLAG("DOMAIN_CODES"),
        FLAG("COMPONENT_CODE_MAP"),
        FLAG("WO_NULL_NON_LEGACY"),
        FLAG("RC_NULL_IFF_NONE"),
        FLAG("NDD_IFF_PLANNED"),
        FLAG("PARTS_QTY_COST_CONSISTENT"),
        FLAG("LABOR_UNEXPLAINED_RATIO"),
    ]
    errors_found = cleaned_df.filter(F.expr(" OR ".join(error_rules))).count()
    if errors_found > 0:
        # Stop before writing Silver if invariant rules fail[cite: 1]
        raise ValueError(
            f"Error gate failed! {errors_found} violations found across invariant rules."
        )

    base_df = cleaned_df.filter(
        ~F.col(FLAG("DUP_EXACT")) & ~F.col(FLAG("VERSION_LATEST"))
    )
    row_quar_df = base_df.filter(F.col(FLAG("EQ_MISSING")))
    valid_silver = base_df.filter(~F.col(FLAG("EQ_MISSING")))

    hash_cols = [
        c
        for c in valid_silver.columns
        if not c.startswith("_") and c not in LINEAGE_COLS
    ]
    valid_silver = (
        valid_silver.withColumn(
            "_silver_row_hash",
            F.sha2(
                F.concat_ws(
                    "|",
                    *[
                        F.coalesce(F.col(c).cast("string"), F.lit("NULL"))
                        for c in hash_cols
                    ],
                ),
                256,
            ),
        )
        .withColumn("_silver_run_id", F.lit(run_id))
        .withColumn("_silver_processed_ts", F.current_timestamp())
    )

    # Generate Quarantine Payloads[cite: 1]
    q_row = row_quar_df.select(
        F.lit("maintenance_history").alias("entity"),
        F.col("maintenance_id").alias("record_id"),
        F.lit("EQ_MISSING").alias("rule_id"),
        F.lit("ROW").alias("quarantine_type"),
        F.lit(None).cast(StringType()).alias("field_name"),
        F.lit("equipment_id null").alias("reason"),
        F.col("_w_source_json").alias("original_values"),
        F.col("record_version"),
        F.col("_batch_id"),
        F.col("_source_file"),
        F.current_timestamp().alias("quarantine_ts"),
        F.current_timestamp().alias("last_seen_ts"),
        F.lit(run_id).alias("last_run_id"),
        F.lit("OPEN").alias("resolution_status"),
        F.lit(None).cast("timestamp").alias("resolved_ts"),
    )

    q_fields = [
        (FLAG("TECH_ID_MISSING"), "TECH_ID_MISSING", "technician_id"),
        (FLAG("TS_END_BEFORE_START"), "TS_END_BEFORE_START", "activity_end_ts"),
        (FLAG("C2"), "NEG_COST_NULLED", "parts_cost_eur"),
        (FLAG("L1"), "SENTINEL_999_NULLED", "labor_hours"),
    ]

    full_quar_df = q_row
    for flag, rule_id, field in q_fields:
        q_f = valid_silver.filter(F.col(flag)).select(
            F.lit("maintenance_history").alias("entity"),
            F.col("maintenance_id").alias("record_id"),
            F.lit(rule_id).alias("rule_id"),
            F.lit("FIELD").alias("quarantine_type"),
            F.lit(field).alias("field_name"),
            F.lit(f"{field} violation").alias("reason"),
            F.concat(
                F.lit(f'{{"{field}": "'), F.col(field).cast(StringType()), F.lit('"}')
            ).alias("original_values"),
            F.col("record_version"),
            F.col("_batch_id"),
            F.col("_source_file"),
            F.current_timestamp().alias("quarantine_ts"),
            F.current_timestamp().alias("last_seen_ts"),
            F.lit(run_id).alias("last_run_id"),
            F.lit("OPEN").alias("resolution_status"),
            F.lit(None).cast("timestamp").alias("resolved_ts"),
        )
        full_quar_df = full_quar_df.unionByName(q_f)

    # Silver Select
    final_silver_cols = [c for c in SOURCE_COLS if c not in LINEAGE_COLS] + [
        "parts_fix_rule",
        "labor_fix_rule",
        "source_file",
        "ingestion_ts",
        "batch_id",
        "source_system",
        "source_system_key",
        "_ingestion_ts",
        "_batch_id",
        "_source_file",
        "_source_system",
        "_record_hash",
        "_silver_run_id",
        "_silver_processed_ts",
        "_silver_row_hash",
    ]
    valid_silver = valid_silver.select(*final_silver_cols)

    # --- Write Operations: Silver Merge ---[cite: 1]
    if spark.catalog.tableExists(silver_table):
        DeltaTable.forName(spark, silver_table).alias("t").merge(
            valid_silver.alias("s"), "t.maintenance_id = s.maintenance_id"
        ).whenMatchedUpdateAll(
            condition="s.record_version > t.record_version OR s._silver_row_hash <> t._silver_row_hash"
        ).whenNotMatchedInsertAll().whenNotMatchedBySourceDelete().execute()
    else:
        valid_silver.write.format("delta").saveAsTable(silver_table)

    # --- Write Operations: Quarantine Merge ---[cite: 1]
    if spark.catalog.tableExists(quar_table):
        if "resolved_ts" not in spark.table(quar_table).columns:
            spark.sql(f"ALTER TABLE {quar_table} ADD COLUMNS (resolved_ts TIMESTAMP)")
        DeltaTable.forName(spark, quar_table).alias("t").merge(
            full_quar_df.alias("s"),
            "t.entity = s.entity AND t.record_id = s.record_id AND t.rule_id = s.rule_id"
            " AND t.field_name <=> s.field_name",
        ).whenMatchedUpdate(
            set={
                "last_seen_ts": "s.last_seen_ts",
                "last_run_id": "s.last_run_id",
                "resolution_status": "'OPEN'",
                "resolved_ts": "NULL",
            }
        ).whenNotMatchedInsertAll().whenNotMatchedBySourceUpdate(
            condition="t.entity = 'maintenance_history' AND t.resolution_status = 'OPEN'",
            set={
                "resolution_status": "'RESOLVED'",
                "resolved_ts": "current_timestamp()",
            },
        ).execute()
    else:
        full_quar_df.write.format("delta").saveAsTable(quar_table)

    # --- Write Operations: DQ Results ---[cite: 1]
    total_bronze = cleaned_df.count()
    exact_dupes = cleaned_df.filter(F.col(FLAG("DUP_EXACT"))).count()
    superseded = cleaned_df.filter(F.col(FLAG("VERSION_LATEST"))).count()

    # base_df is the dataset AFTER deduplication (9,800 rows constraint)
    total_base = base_df.count()
    total_silver = valid_silver.count()
    total_row_quar = row_quar_df.count()
    records_failed_flow = total_bronze - (
        total_silver + exact_dupes + superseded + total_row_quar
    )

    audit_records = [
        {
            "table_name": config["tables"]["silver"],
            "rule_name": "ROW_FLOW:ALL",
            "dimension": "Completeness",
            "severity": "Warn",
            "description": "Every Bronze row has exactly one outcome",
            "records_evaluated": total_bronze,
            "records_failed": records_failed_flow,
            "failure_rate": float(records_failed_flow) / total_bronze
            if total_bronze > 0
            else 0.0,
            "run_ts": datetime.now(UTC),
            "run_id": run_id,
        }
    ]

    rule_flags = [
        c.replace("_f_", "") for c in cleaned_df.columns if c.startswith("_f_")
    ]

    # 1. Count Raw Rules on cleaned_df (10,000 rows)
    raw_rules = ["TYPE_CAST", "DUP_EXACT", "VERSION_LATEST"]
    raw_exprs = [
        F.sum(F.col(f"_f_{rule}").cast("int")).alias(rule)
        for rule in raw_rules
        if rule in rule_flags
    ]
    if raw_exprs:
        raw_counts = cleaned_df.agg(*raw_exprs).collect()[0].asDict()
        for rule_name, count in raw_counts.items():
            sev, dim, desc = config["rule_meta"][rule_name]
            audit_records.append(
                {
                    "table_name": config["tables"]["silver"],
                    "rule_name": rule_name,
                    "dimension": dim,
                    "severity": sev,
                    "description": desc,
                    "records_evaluated": total_bronze,
                    "records_failed": count or 0,
                    "failure_rate": float(count or 0) / total_bronze
                    if total_bronze > 0
                    else 0.0,
                    "run_ts": datetime.now(UTC),
                    "run_id": run_id,
                }
            )

    # 2. Count Business Rules on base_df (9,800 rows after deduplication)
    bus_rules = [
        r for r in rule_flags if r not in raw_rules and r in config["rule_meta"]
    ]
    bus_exprs = [
        F.sum(F.col(f"_f_{rule}").cast("int")).alias(rule) for rule in bus_rules
    ]
    if bus_exprs:
        bus_counts = base_df.agg(*bus_exprs).collect()[0].asDict()
        for rule_name, count in bus_counts.items():
            sev, dim, desc = config["rule_meta"][rule_name]
            audit_records.append(
                {
                    "table_name": config["tables"]["silver"],
                    "rule_name": rule_name,
                    "dimension": dim,
                    "severity": sev,
                    "description": desc,
                    "records_evaluated": total_base,
                    "records_failed": count or 0,
                    "failure_rate": float(count or 0) / total_base
                    if total_base > 0
                    else 0.0,
                    "run_ts": datetime.now(UTC),
                    "run_id": run_id,
                }
            )

    # Respect existing table schema: total_records_processed, valid_records,
    # quarantined_records, pipeline_run_ts, dataset_name
    audit_df = spark.createDataFrame(
        [(total_bronze, total_silver, total_row_quar, datetime.now(UTC), "maintenance_history")],
        schema="total_records_processed LONG, valid_records LONG, quarantined_records LONG, pipeline_run_ts TIMESTAMP, dataset_name STRING",
    )
    audit_df.write.format("delta").mode("append").saveAsTable(audit_table)



run_id = f"mh-{datetime.now(UTC):%Y%m%dT%H%M%S}-{uuid.uuid4().hex[:6]}"

bronze_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.test_bronze_maintenance_history")

cleaned_df = clean_maintenance_history(bronze_df, CONFIG)
create_silver_tables(cleaned_df, CATALOG_NAME, SCHEMA_NAME, run_id, CONFIG, spark)




## Silver WorkOrders Ingestion 

In [0]:
# Databricks notebook source
"""Silver layer for work orders: bronze_work_orders -> silver_work_orders.

Implements silver_work_orders_rules.md. Layout:
    1. CONFIG                 parameters, codes, rules, Silver schema
    2. clean_worker_ord       cleaning tools: Bronze rows -> cleaned rows with _f_<RULE> flags
    3. creating_silver_table  rule counts, error gate, MERGE into Silver and quarantine
    4. main                   widgets, read Bronze, run parts 2 and 3
Needs Spark 3.5+ (Databricks Runtime 14.3+): try_to_timestamp, nullif.
"""



# ======== 1. CONFIG ===============================================================================
TABLE_NAMES = {
    "bronze": "test_bronze_work_orders",  # the only input
    "silver": "test_silver_work_orders",
    "quarantine": "test_silver_dq_quarantine",
    "audit": "test_dq_results",
}
QUARANTINE_ENTITY = "work_orders"
BRONZE_LOADED_STATUS = "SUCCESS"
BRONZE_COLUMN_MAP = {
    "batch_id": "_batch_id",
    "source_batch_id": "batch_id",
    "ingestion_timestamp": "_ingestion_ts",
    "source_filename": "_source_file",
    "record_hash": "_record_hash",
    "load_status": "_load_status",
}
TIMESTAMP_PATTERN = "yyyy-MM-dd'T'HH:mm:ss'Z'"
NULL_MARKER = "<NULL>"
# Labour and currency (Decimal or str, never float)
STANDARD_LABOUR_RATE_EUR_H = Decimal(
    "55"
)  # domain assumption for L1/L2/LC1, never stored
LABOUR_SENTINELS = [9999]
LABOUR_MATCH_TOLERANCE_H = Decimal("0.01")  # EUR rows
COST_MATCH_TOLERANCE_EUR = Decimal("0.01")  # EUR rows
NON_EUR_COST_TOLERANCE_PCT = 1.5  # GBP/USD rows: the fixed rate is approximate
FX_RATE_SOURCE = "FIXED_RATE"  # OPEN: confirm official rates with the instructor
FX_RATES_TO_EUR = {
    "EUR": Decimal("1.0000"),
    "GBP": Decimal("1.1700"),
    "USD": Decimal("0.9120"),
}

SLA_HOURS = {"P1": 24, "P2": 72, "P3": 168, "P4": 336}
PLANNED_WO_TYPES = ["Preventive", "Inspection"]
UNPLANNED_WO_TYPES = ["Corrective"]
RECOVERY_RECORD_VERSION = 1  # closed_ts is only recovered on version 1

CODES = {
    "wo_type": ["Preventive", "Corrective", "Inspection", "Predictive"],
    "priority": ["P1", "P2", "P3", "P4"],
    "status": ["Open", "In Progress", "Closed", "Cancelled"],
    "downtime_type": ["planned", "unplanned"],
    "currency": ["EUR", "GBP", "USD"],
    "requested_by_role": [
        "Planner",
        "Operator",
        "Shift Supervisor",
        "Condition Monitoring",
    ],
    "cancellation_reason": [
        "Scope merged into next PM",
        "Raised in error",
        "Asset not available - production priority",
    ],
    "failure_mode_code": [
        "NONE",
        "BRG-WEAR",
        "MISALIGN",
        "IMBAL",
        "OVERHEAT",
        "LUB-FAIL",
        "WIND-INS",
        "SEAL-LEAK",
        "CAVIT",
        "VALVE-FAIL",
        "GEAR-WEAR",
        "CLOG",
    ],
    "root_cause_code": [
        "RC-WEAR",
        "RC-CONTAM",
        "RC-LUB",
        "RC-INST",
        "RC-OPER",
        "RC-MAINT",
        "RC-DESIGN",
        "RC-MATL",
    ],
}
CHECK_ONLY_WHEN_SET = [
    "downtime_type",
    "root_cause_code",
    "cancellation_reason",
    "work_order_id",
]
PATTERNS = {
    "work_order_id": r"^WO-[0-9]{5}$",
    "assigned_technician_id": r"^TECH-[0-9]{2}$",
}
TRIGGER_PATTERN = r"^SE-[0-9]{6}$"
FAILURE_MODE_DESC = {
    "NONE": "No failure (planned work)",
    "BRG-WEAR": "Bearing wear",
    "MISALIGN": "Shaft misalignment",
    "IMBAL": "Rotor imbalance",
    "OVERHEAT": "Overheating",
    "LUB-FAIL": "Lubrication failure",
    "WIND-INS": "Stator winding insulation breakdown",
    "SEAL-LEAK": "Mechanical seal leakage",
    "CAVIT": "Cavitation",
    "VALVE-FAIL": "Valve failure",
    "GEAR-WEAR": "Gear tooth wear",
    "CLOG": "Blocked suction / plugged strainer",
}

# Columns: 27 business + 5 source lineage + Bronze audit columns (after renaming)
BUSINESS_COLS = [
    "work_order_id",
    "equipment_id",
    "wo_type",
    "priority",
    "failure_mode_code",
    "failure_mode_desc",
    "description",
    "requested_by_role",
    "assigned_technician_id",
    "created_ts",
    "due_ts",
    "actual_start_ts",
    "closed_ts",
    "downtime_start_ts",
    "downtime_end_ts",
    "downtime_minutes",
    "downtime_type",
    "status",
    "cancellation_reason",
    "root_cause_code",
    "triggering_event_id",
    "labor_hours",
    "labor_cost",
    "parts_cost",
    "currency",
    "last_modified_ts",
    "record_version",
]
LINEAGE_COLS = [
    "source_file",
    "ingestion_ts",
    "batch_id",
    "source_system",
    "source_system_key",
]
BRONZE_AUDIT_COLS = [
    "_batch_id",
    "_ingestion_ts",
    "_source_file",
    "_source_system",
    "_record_hash",
    "_load_status",
]
REQUIRED_COLS = BUSINESS_COLS + LINEAGE_COLS + BRONZE_AUDIT_COLS
TIMESTAMP_COLS = [
    "created_ts",
    "due_ts",
    "actual_start_ts",
    "closed_ts",
    "downtime_start_ts",
    "downtime_end_ts",
    "last_modified_ts",
    "ingestion_ts",
    "_ingestion_ts",
]
EVENT_TS_COLS = [
    "created_ts",
    "actual_start_ts",
    "closed_ts",
    "downtime_start_ts",
    "downtime_end_ts",
    "last_modified_ts",
]  # due_ts is a deadline, not an event
DOWNTIME_COLS = [
    "downtime_start_ts",
    "downtime_end_ts",
    "downtime_minutes",
    "downtime_type",
]
NUMERIC_TYPES = {
    "labor_hours": "decimal(8,2)",
    "labor_cost": "decimal(12,2)",
    "parts_cost": "decimal(12,2)",
    "downtime_minutes": "int",
    "record_version": "int",
}

# Value written in the *_fix_rule columns for each rule that changed a value
TS_FIX_LABELS = {
    "TS_CLOSED_RECOVERED": "CLOSED_TS_FROM_LAST_MODIFIED",
    "TS_CLOSED_UNRECOVERABLE": "CLOSED_TS_UNRECOVERABLE",
}
LABOUR_FIX_LABELS = {
    "L1": "SENTINEL_RECOVERED_FROM_COST",
    "L2": "NEGATIVE_ABS_VERIFIED_BY_COST",
    "L3": "LABOR_HOURS_UNRECOVERABLE",
    "LC1": "NEG_COST_RECOVERED_FROM_HOURS",
}
PARTS_FIX_LABELS = {"P1": "NEG_PARTS_NULLED"}

# rule -> (dimension, severity, description). Adding a rule = one line here + one _f_<rule> boolean
# column in a cleaning step (except DEDUP_RULES). ERROR rules above 0 stop the run before any write.
DEDUP_RULES = ["DUP_EXACT", "VERSION_LATEST"]
RULES = {
    "DUP_EXACT": ("uniqueness", "DROP", "Exact duplicate of the business columns"),
    "VERSION_LATEST": ("uniqueness", "DROP", "Superseded record version"),
    "TYPE_CAST": ("validity", "ERROR", "Non-empty value that does not parse"),
    "MISSING_WORK_ORDER_ID": (
        "completeness",
        "ROW_QUARANTINE",
        "work_order_id missing",
    ),
    "MISSING_CREATED_TS": ("completeness", "ROW_QUARANTINE", "created_ts missing"),
    "SOURCE_KEY_FORMAT": (
        "validity",
        "ERROR",
        "source_system_key <> system|work_order_id",
    ),
    "DOMAIN_CODES": (
        "validity",
        "ERROR",
        "Code outside its list or ID with a wrong format",
    ),
    "FM_DESC_MAP": (
        "consistency",
        "ERROR",
        "failure_mode_desc does not match failure_mode_code",
    ),
    "FM_NONE_IFF_PLANNED_TYPE": (
        "consistency",
        "ERROR",
        "NONE <=> Preventive or Inspection",
    ),
    "RC_NULL_IFF_NOT_FAILURE_CLOSED": (
        "consistency",
        "ERROR",
        "root cause null <=> planned/open",
    ),
    "DUE_MATCHES_PRIORITY": (
        "consistency",
        "ERROR",
        "due_ts is not created_ts + SLA of priority",
    ),
    "CANCEL_REASON_IFF_CANCELLED": ("consistency", "ERROR", "reason set <=> Cancelled"),
    "STATUS_LIFECYCLE": (
        "consistency",
        "ERROR",
        "Field pattern does not match the status",
    ),
    "TS_CLOSED_RECOVERED": (
        "consistency",
        "FIX",
        "closed_ts < created_ts, from last_modified",
    ),
    "TS_CLOSED_UNRECOVERABLE": (
        "consistency",
        "FIELD_QUARANTINE",
        "closed_ts wrong, no safe fix",
    ),
    "TS_ORDER": ("consistency", "ERROR", "created/start/downtime end after closed_ts"),
    "TS_MODIFIED_BEFORE_CLOSE": (
        "consistency",
        "ERROR",
        "last_modified_ts before closed_ts",
    ),
    "TS_AFTER_EXTRACT": ("validity", "WARN", "Event timestamp after ingestion_ts"),
    "START_BEFORE_CREATED": (
        "consistency",
        "WARN",
        "Emergency start before work order creation",
    ),
    "FX_UNAVAILABLE": (
        "validity",
        "FIELD_QUARANTINE",
        "No fixed rate for the currency",
    ),
    "L1": ("validity", "FIX", "Labour hours sentinel recovered from labor_cost"),
    "L2": ("validity", "FIX", "Negative labour hours made positive, verified by cost"),
    "L3": ("validity", "FIELD_QUARANTINE", "Labour hours not recoverable"),
    "LC1": ("validity", "FIX", "Negative labour cost recovered from hours"),
    "LABOR_COST_CONSISTENT": (
        "consistency",
        "ERROR",
        "labor_cost is not hours x standard rate",
    ),
    "P1": (
        "validity",
        "FIELD_QUARANTINE",
        "Negative parts cost nulled (recovery is Gold)",
    ),
    "EQ_MISSING": (
        "completeness",
        "FIELD_QUARANTINE",
        "equipment_id missing, kept null (Gold)",
    ),
    "DT_COMPLETE": ("completeness", "ERROR", "Downtime fields not all set or all null"),
    "DT_MINUTES_MATCH": (
        "consistency",
        "ERROR",
        "downtime_minutes differs from end - start",
    ),
    "DT_TYPE_MATCHES_WO_TYPE": ("consistency", "ERROR", "unplanned <=> Corrective"),
    "DT_TIMING": ("consistency", "WARN", "Downtime start does not fit its type"),
    "DT_OVERLAP": (
        "consistency",
        "FLAG",
        "Downtime overlaps an earlier one on the same asset",
    ),
    "TRIGGER_FORMAT": ("validity", "ERROR", "Sensor trigger format, Predictive only"),
}

# rule -> (type, [(field set to null, column holding its value as delivered)]).
# ROW rules keep the row out of Silver (field_name stays null, original_values = whole row).
QUARANTINE_RULES = {
    "MISSING_WORK_ORDER_ID": ("ROW", []),
    "MISSING_CREATED_TS": ("ROW", []),
    "EQ_MISSING": ("FIELD", [("equipment_id", "equipment_id")]),
    "P1": ("FIELD", [("parts_cost_eur", "parts_cost")]),
    "TS_CLOSED_UNRECOVERABLE": ("FIELD", [("closed_ts", "_orig_closed_ts")]),
    "L3": ("FIELD", [("labor_hours", "_orig_labor_hours")]),
    "FX_UNAVAILABLE": (
        "FIELD",
        [("labor_cost_eur", "labor_cost"), ("parts_cost_eur", "parts_cost")],
    ),
}
ROW_QUARANTINE_RULES = [
    rule for rule, (kind, _) in QUARANTINE_RULES.items() if kind == "ROW"
]

# Silver table: (column, type, comment). Also drives casting, the row hash and the DDL.
SILVER_SCHEMA = [
    ("work_order_id", "string", "WO-nnnnn, primary key"),
    (
        "equipment_id",
        "string",
        "Asset as delivered; null when missing in source (EQ_MISSING)",
    ),
    ("wo_type", "string", "Preventive, Corrective, Inspection, Predictive"),
    ("priority", "string", "P1 to P4"),
    ("failure_mode_code", "string", "NONE = planned work"),
    ("failure_mode_desc", "string", "Description of failure_mode_code"),
    ("description", "string", "Work order text"),
    (
        "requested_by_role",
        "string",
        "Planner, Operator, Shift Supervisor, Condition Monitoring",
    ),
    ("assigned_technician_id", "string", "TECH-nn, pseudonymous"),
    ("created_ts", "timestamp", "UTC"),
    ("due_ts", "timestamp", "created_ts + SLA of the priority"),
    ("actual_start_ts", "timestamp", "Null = not started (Open, Cancelled)"),
    ("closed_ts", "timestamp", "Null = not finished; fixed per TS_CLOSED_RECOVERED"),
    ("downtime_start_ts", "timestamp", "Null = no downtime"),
    ("downtime_end_ts", "timestamp", "Null = no downtime or still running"),
    ("downtime_minutes", "int", "Null = no downtime or still running"),
    ("downtime_type", "string", "planned or unplanned"),
    ("status", "string", "Open, In Progress, Closed, Cancelled"),
    ("cancellation_reason", "string", "Set only when Cancelled"),
    ("root_cause_code", "string", "Null = planned work or not yet closed"),
    (
        "triggering_event_id",
        "string",
        "Sensor event that raised a predictive work order",
    ),
    ("labor_hours", "decimal(8,2)", "Booked person-hours, cleaned per L1/L2"),
    ("labor_cost_eur", "decimal(12,2)", "Labour cost in EUR at the fixed rate"),
    (
        "parts_cost_eur",
        "decimal(12,2)",
        "Parts cost EUR; null if not closed or negative",
    ),
    ("currency", "string", "Source currency"),
    ("fx_rate_to_eur", "decimal(8,4)", "Fixed rate applied (1 for EUR)"),
    ("last_modified_ts", "timestamp", "Last change in CMMS, UTC"),
    ("record_version", "int", "Version kept"),
    (
        "ts_fix_rule",
        "string",
        "CLOSED_TS_FROM_LAST_MODIFIED, CLOSED_TS_UNRECOVERABLE or UNCHANGED",
    ),
    (
        "labor_fix_rule",
        "string",
        "Labour rules applied, joined by semicolon, or UNCHANGED",
    ),
    ("parts_fix_rule", "string", "NEG_PARTS_NULLED or UNCHANGED"),
    ("equipment_resolution", "string", "SOURCE or MISSING"),
    (
        "dq_downtime_overlap",
        "boolean",
        "Downtime overlaps an earlier one on the same asset",
    ),
    ("source_file", "string", "Source lineage"),
    ("ingestion_ts", "timestamp", "Source lineage"),
    ("batch_id", "string", "Source batch"),
    ("source_system", "string", "Source lineage"),
    ("source_system_key", "string", "source_system|work_order_id"),
    ("_ingestion_ts", "timestamp", "Bronze ingestion time of the kept row"),
    ("_batch_id", "string", "Bronze micro-batch of the kept row"),
    ("_source_file", "string", "Bronze source file of the kept row"),
    ("_source_system", "string", "Bronze source system of the kept row"),
    ("_record_hash", "string", "Bronze record hash, joins back to Bronze"),
    ("_silver_run_id", "string", "Run metadata"),
    ("_silver_processed_ts", "timestamp", "Run metadata"),
    ("_silver_row_hash", "string", "SHA-256 of business, fix-rule and flag columns"),
]
ROW_HASH_COLS = [
    n for n, _, _ in SILVER_SCHEMA if not n.startswith("_") and n not in LINEAGE_COLS
]

# Shared silver_dq_quarantine schema: identical in silver_maintenance.py and silver_sensor_event.py
# (spec: context/silver_max/silver_maintenance_rules.md 6.3). Change all three together.
QUARANTINE_DDL = """CREATE TABLE IF NOT EXISTS {table} (
    entity STRING, record_id STRING, rule_id STRING, quarantine_type STRING, field_name STRING,
    reason STRING, original_values STRING, record_version INT, _batch_id STRING,
    _source_file STRING, quarantine_ts TIMESTAMP, last_seen_ts TIMESTAMP, last_run_id STRING,
    resolution_status STRING, resolved_ts TIMESTAMP
) USING DELTA"""
AUDIT_SCHEMA = (
    "table_name string, rule_name string, dimension string, severity string, "
    "description string, records_evaluated long, records_failed long, "
    "failure_rate double, run_ts timestamp, run_id string"
)


class DataQualityError(Exception):
    """Raised when the Bronze input or an ERROR rule makes it unsafe to write Silver."""


def qualify_tables(catalog: str, schema: str) -> dict[str, str]:
    """Fully qualified table names; catalog and schema come from notebook widgets."""
    return {key: f"{catalog}.{schema}.{name}" for key, name in TABLE_NAMES.items()}


# ======== 2. clean_worker_ord: the cleaning tools =================================================
def _true(condition: Column) -> Column:
    """A null condition counts as False, so flags are never null."""
    return F.coalesce(condition, F.lit(False))


def _set_flags(df: DataFrame, conditions: dict[str, Column]) -> DataFrame:
    """Add one boolean _f_<RULE> column per rule."""
    return df.select(
        "*", *[_true(cond).alias(f"_f_{rule}") for rule, cond in conditions.items()]
    )


def _row_hash(columns: list[str]) -> Column:
    """SHA-256 over the columns, nulls replaced by a fixed marker."""
    parts = [F.coalesce(F.col(c).cast("string"), F.lit(NULL_MARKER)) for c in columns]
    return F.sha2(F.concat_ws("\x1f", *parts), 256)


def _lookup(column: Column, mapping: dict) -> Column:
    """CASE WHEN chain, null for unknown keys (a map lookup would fail under ANSI mode)."""
    expression = F.lit(None)
    for key, value in mapping.items():
        expression = F.when(column == key, F.lit(value)).otherwise(expression)
    return expression


def _fix_rule_column(labels: dict[str, str]) -> Column:
    """Names of the fixes applied to a row joined by semicolon, or UNCHANGED."""
    applied = [
        F.when(F.col(f"_f_{rule}"), F.lit(name)) for rule, name in labels.items()
    ]
    joined = F.concat_ws(";", *applied)
    return F.when(joined == "", F.lit("UNCHANGED")).otherwise(joined)


def _to_eur(amount: str) -> Column:
    """Local amount x fixed rate, HALF_UP to 2 places, decimal arithmetic."""
    return F.round(F.col(amount) * F.col("fx_rate_to_eur"), 2).cast("decimal(12,2)")


def _within_tolerance(
    actual: Column, expected: Column, absolute_tolerance: Decimal
) -> Column:
    """EUR rows must match to the cent, GBP/USD rows within NON_EUR_COST_TOLERANCE_PCT."""
    tolerance = F.when(F.col("currency") == "EUR", F.lit(absolute_tolerance)).otherwise(
        F.abs(expected) * (NON_EUR_COST_TOLERANCE_PCT / 100)
    )
    return F.abs(actual - expected) <= tolerance


def _downtime_set_count() -> Column:
    return reduce(add, [F.col(c).isNotNull().cast("int") for c in DOWNTIME_COLS])


def _is_running_downtime() -> Column:
    """In Progress work order whose downtime has started but not ended: valid."""
    return (
        (F.col("status") == "In Progress")
        & F.col("downtime_start_ts").isNotNull()
        & F.col("downtime_type").isNotNull()
        & F.col("downtime_end_ts").isNull()
        & F.col("downtime_minutes").isNull()
    )


def adapt_bronze_columns(df: DataFrame) -> DataFrame:
    """Rename Bronze audit columns (same map as maintenance history), copy _source_system."""
    df = df.select(*[F.col(c).alias(BRONZE_COLUMN_MAP.get(c, c)) for c in df.columns])
    if "source_system" in df.columns:
        df = df.withColumn("_source_system", F.col("source_system"))
    return df


def check_contracts(df: DataFrame) -> None:
    """SCHEMA_CONTRACT: every required column must be present."""
    missing = sorted(set(REQUIRED_COLS) - set(df.columns))
    if missing:
        raise DataQualityError(f"SCHEMA_CONTRACT: Bronze columns missing: {missing}")


def filter_loaded(df: DataFrame) -> DataFrame:
    """Keep rows with _load_status = SUCCESS, print the skipped count, stop on zero rows."""
    loaded = df.filter(F.trim(F.col("_load_status")) == BRONZE_LOADED_STATUS)
    total, kept = df.count(), loaded.count()
    print(
        f"Bronze rows read: {total:,}; skipped (load_status <> {BRONZE_LOADED_STATUS}): "
        f"{total - kept:,}"
    )
    if kept == 0:
        raise DataQualityError("No Bronze row with a loaded status: nothing to do.")
    return loaded


def standardise(df: DataFrame) -> DataFrame:
    """Trim, empty -> null, keep case, parse types without throwing (TYPE_CAST flags failures)."""
    dtypes = dict(df.dtypes)
    columns, cast_failures = [], []
    for name in REQUIRED_COLS:
        text = F.nullif(F.trim(F.col(name).cast("string")), F.lit(""))
        if name in TIMESTAMP_COLS:
            parsed = (
                F.col(name)
                if dtypes[name] == "timestamp"
                else F.try_to_timestamp(text, F.lit(TIMESTAMP_PATTERN))
            )
        elif name in NUMERIC_TYPES:
            sql_type = NUMERIC_TYPES[name]
            parsed = F.expr(
                f"try_cast(nullif(trim(cast(`{name}` as string)), '') as {sql_type})"
            )
        else:
            columns.append(text.alias(name))
            continue
        columns.append(parsed.alias(name))
        cast_failures.append(text.isNotNull() & parsed.isNull())
    return df.select(*columns, reduce(or_, cast_failures).alias("_f_TYPE_CAST"))


def deduplicate(df: DataFrame) -> DataFrame:
    """Flag exact duplicates and superseded versions. Drops nothing: the caller filters."""
    df = df.withColumn("_dup_hash", _row_hash(BUSINESS_COLS))
    first_copy = Window.partitionBy("_dup_hash").orderBy(
        "_ingestion_ts", "_source_file", "_record_hash"
    )
    df = df.withColumn("_f_DUP_EXACT", F.row_number().over(first_copy) > 1)
    # Never order by _ingestion_ts alone: both versions can share one ingestion time.
    version_order = Window.partitionBy(
        F.coalesce("work_order_id", "_record_hash")
    ).orderBy(
        F.col("_f_DUP_EXACT").asc(),
        F.col("record_version").desc(),
        F.col("last_modified_ts").desc(),
        F.col("_ingestion_ts").desc(),
        F.col("_record_hash").asc(),
    )
    is_older = F.row_number().over(version_order) > 1
    return df.withColumn("_f_VERSION_LATEST", ~F.col("_f_DUP_EXACT") & is_older)


def fix_timestamps(df: DataFrame) -> DataFrame:
    """closed_ts before created_ts: take last_modified_ts, but only when it passes every guard."""
    modified = F.col("last_modified_ts")
    closed_before_created = F.col("closed_ts") < F.col("created_ts")
    recoverable = _true(
        (F.col("record_version") == RECOVERY_RECORD_VERSION)
        & (modified >= F.col("created_ts"))
        & (F.col("actual_start_ts").isNull() | (modified >= F.col("actual_start_ts")))
        & (F.col("downtime_end_ts").isNull() | (modified >= F.col("downtime_end_ts")))
    )
    df = df.withColumn(
        "_orig_closed_ts", F.col("closed_ts")
    )  # kept for quarantine original_values
    df = _set_flags(
        df,
        {
            "TS_CLOSED_RECOVERED": closed_before_created & recoverable,
            "TS_CLOSED_UNRECOVERABLE": closed_before_created & ~recoverable,
        },
    )
    fixed_closed = (
        F.when(F.col("_f_TS_CLOSED_RECOVERED"), modified)
        .when(F.col("_f_TS_CLOSED_UNRECOVERABLE"), F.lit(None).cast("timestamp"))
        .otherwise(F.col("closed_ts"))
    )
    return df.withColumn("closed_ts", fixed_closed).withColumn(
        "ts_fix_rule", _fix_rule_column(TS_FIX_LABELS)
    )


def convert_currency(df: DataFrame) -> DataFrame:
    """Attach the fixed rate from FX_RATES_TO_EUR and convert both amounts to EUR."""
    rate = _lookup(F.col("currency"), FX_RATES_TO_EUR).cast("decimal(8,4)")
    df = df.withColumn("fx_rate_to_eur", rate)
    df = _set_flags(df, {"FX_UNAVAILABLE": F.col("fx_rate_to_eur").isNull()})
    return df.withColumn("labor_cost_eur", _to_eur("labor_cost")).withColumn(
        "parts_cost_eur", _to_eur("parts_cost")
    )


def fix_labour(df: DataFrame) -> DataFrame:
    """L1/L2/L3 repair labor_hours, LC1 repairs labor_cost, all from the row's own data."""
    df = df.withColumn(
        "_orig_labor_hours", F.col("labor_hours")
    )  # kept for quarantine original_values
    hours, cost, rate = (
        F.col("labor_hours"),
        F.col("labor_cost"),
        F.col("fx_rate_to_eur"),
    )
    standard = F.lit(STANDARD_LABOUR_RATE_EUR_H)
    hours_from_cost = F.round(cost * rate / standard, 2)
    is_sentinel, is_negative, cost_is_valid = (
        hours.isin(LABOUR_SENTINELS),
        hours < 0,
        cost > 0,
    )
    # Bad hours are found by sentinel list and cost consistency, never by a "> 24 h" threshold.
    df = _set_flags(
        df,
        {
            "L1": is_sentinel & cost_is_valid,
            "L2": is_negative
            & cost_is_valid
            & _within_tolerance(
                F.abs(hours), hours_from_cost, LABOUR_MATCH_TOLERANCE_H
            ),
        },
    )
    df = _set_flags(
        df, {"L3": (is_sentinel | is_negative) & ~F.col("_f_L1") & ~F.col("_f_L2")}
    )
    fixed_hours = (
        F.when(F.col("_f_L1"), hours_from_cost)
        .when(F.col("_f_L2"), F.abs(hours))
        .when(F.col("_f_L3"), F.lit(None))
        .otherwise(hours)
    )
    df = df.withColumn("labor_hours", fixed_hours.cast("decimal(8,2)"))
    df = _set_flags(df, {"LC1": (cost < 0) & (F.col("labor_hours") > 0)})
    cost_from_hours = F.round(F.col("labor_hours") * standard / rate, 2)
    fixed_cost = F.when(F.col("_f_LC1"), cost_from_hours).otherwise(cost)
    return (
        df.withColumn("labor_cost", fixed_cost.cast("decimal(12,2)"))
        .withColumn("labor_cost_eur", _to_eur("labor_cost"))  # LC1 may change the cost
        .withColumn("labor_fix_rule", _fix_rule_column(LABOUR_FIX_LABELS))
    )


def fix_parts(df: DataFrame) -> DataFrame:
    """P1: negative parts cost is nulled. No abs() and no +10: Gold recovers it from activities."""
    df = _set_flags(df, {"P1": F.col("parts_cost") < 0})
    nulled = F.when(F.col("_f_P1"), F.lit(None)).otherwise(F.col("parts_cost_eur"))
    return df.withColumn("parts_cost_eur", nulled.cast("decimal(12,2)")).withColumn(
        "parts_fix_rule", _fix_rule_column(PARTS_FIX_LABELS)
    )


def resolve_equipment(df: DataFrame) -> DataFrame:
    """Missing equipment stays null. Registration is not checked here: that is Gold."""
    df = _set_flags(df, {"EQ_MISSING": F.col("equipment_id").isNull()})
    resolution = F.when(F.col("_f_EQ_MISSING"), "MISSING").otherwise("SOURCE")
    return df.withColumn("equipment_resolution", resolution)


def _violates_domain(column: str, is_valid: Column) -> Column:
    """Value outside its domain. Empty values only count for columns that are required."""
    if column in CHECK_ONLY_WHEN_SET:
        return F.col(column).isNotNull() & ~is_valid
    return ~_true(is_valid)


def check_keys_and_codes(df: DataFrame) -> DataFrame:
    """Key presence, source key, code lists, ID formats, sensor trigger."""
    domain_checks = [(c, F.col(c).isin(values)) for c, values in CODES.items()] + [
        (c, F.col(c).rlike(pattern)) for c, pattern in PATTERNS.items()
    ]
    trigger = F.col("triggering_event_id")
    source_key = F.concat_ws("|", "source_system", "work_order_id")
    return _set_flags(
        df,
        {
            "MISSING_WORK_ORDER_ID": F.col("work_order_id").isNull(),
            "MISSING_CREATED_TS": F.col("created_ts").isNull(),
            "SOURCE_KEY_FORMAT": F.col("work_order_id").isNotNull()
            & ~F.col("source_system_key").eqNullSafe(source_key),
            "DOMAIN_CODES": reduce(
                or_, [_violates_domain(c, ok) for c, ok in domain_checks]
            ),
            "TRIGGER_FORMAT": trigger.isNotNull()
            & (~trigger.rlike(TRIGGER_PATTERN) | (F.col("wo_type") != "Predictive")),
        },
    )


def check_consistency(df: DataFrame) -> DataFrame:
    """Failure mode, root cause, SLA and cancellation rules (5.3)."""
    no_failure = F.col("failure_mode_code") == "NONE"
    expected_desc = _lookup(F.col("failure_mode_code"), FAILURE_MODE_DESC)
    sla_seconds = _lookup(
        F.col("priority"), {p: h * 3600 for p, h in SLA_HOURS.items()}
    )
    expected_due = F.unix_timestamp("created_ts") + sla_seconds
    root_cause_is_null = F.col("root_cause_code").isNull()
    has_reason = F.col("cancellation_reason").isNotNull()
    is_cancelled = F.col("status") == "Cancelled"
    return _set_flags(
        df,
        {
            "FM_DESC_MAP": ~F.col("failure_mode_desc").eqNullSafe(expected_desc),
            "FM_NONE_IFF_PLANNED_TYPE": ~no_failure.eqNullSafe(
                F.col("wo_type").isin(PLANNED_WO_TYPES)
            ),
            "RC_NULL_IFF_NOT_FAILURE_CLOSED": ~root_cause_is_null.eqNullSafe(
                no_failure | (F.col("status") != "Closed")
            ),
            "DUE_MATCHES_PRIORITY": F.unix_timestamp("due_ts") != expected_due,
            "CANCEL_REASON_IFF_CANCELLED": ~has_reason.eqNullSafe(is_cancelled),
        },
    )


def check_timestamps(df: DataFrame) -> DataFrame:
    """Checks that run after the closure fix (5.5)."""
    closed, created, start = (
        F.col("closed_ts"),
        F.col("created_ts"),
        F.col("actual_start_ts"),
    )
    after_extract = reduce(
        or_, [F.col(c) > F.col("ingestion_ts") for c in EVENT_TS_COLS]
    )
    return _set_flags(
        df,
        {
            "TS_ORDER": (closed < created)
            | (closed < start)
            | (F.col("downtime_end_ts") > closed),
            "TS_MODIFIED_BEFORE_CLOSE": F.col("last_modified_ts") < closed,
            "TS_AFTER_EXTRACT": after_extract,
            "START_BEFORE_CREATED": start < created,  # a warning, not a defect
        },
    )


def check_downtime(df: DataFrame) -> DataFrame:
    """Downtime checks and the overlap flag. Overlaps are flagged, never quarantined (5.9)."""
    start, end = F.col("downtime_start_ts"), F.col("downtime_end_ts")
    dt_type = F.col("downtime_type")
    minutes_calc = F.round((F.unix_timestamp(end) - F.unix_timestamp(start)) / 60)
    minutes_before_creation = (
        F.unix_timestamp("created_ts") - F.unix_timestamp(start)
    ) / 60
    # Running maximum of earlier end times per asset, not only the previous row.
    earlier = (
        Window.partitionBy("equipment_id")
        .orderBy("downtime_start_ts", "work_order_id")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    type_mismatch = (dt_type == "unplanned") != F.col("wo_type").isin(
        UNPLANNED_WO_TYPES
    )
    df = _set_flags(
        df,
        {
            "DT_COMPLETE": ~(_downtime_set_count().isin(0, 4) | _is_running_downtime()),
            "DT_MINUTES_MATCH": F.abs(F.col("downtime_minutes") - minutes_calc) > 1,
            "DT_TYPE_MATCHES_WO_TYPE": dt_type.isNotNull() & type_mismatch,
            "DT_TIMING": (
                (dt_type == "unplanned") & ~minutes_before_creation.between(0, 30)
            )
            | ((dt_type == "planned") & (start < F.col("created_ts"))),
            "DT_OVERLAP": F.col("equipment_id").isNotNull()
            & (start < F.max(end).over(earlier)),
        },
    )
    return df.withColumn("dq_downtime_overlap", F.col("_f_DT_OVERLAP"))


def check_labour_consistency(df: DataFrame) -> DataFrame:
    """Every Closed row: labor_cost = hours x standard rate / fixed rate (within tolerance)."""
    standard = F.lit(STANDARD_LABOUR_RATE_EUR_H)
    expected = F.round(F.col("labor_hours") * standard / F.col("fx_rate_to_eur"), 2)
    both_set = F.col("labor_hours").isNotNull() & F.col("labor_cost").isNotNull()
    consistent = _within_tolerance(
        F.col("labor_cost"), expected, COST_MATCH_TOLERANCE_EUR
    )
    is_closed = F.col("status") == "Closed"
    return _set_flags(df, {"LABOR_COST_CONSISTENT": is_closed & both_set & ~consistent})


def check_status_lifecycle(df: DataFrame) -> DataFrame:
    """Strict field pattern per status. Nulls are valid and never filled with placeholders."""
    status, started = F.col("status"), F.col("actual_start_ts").isNotNull()
    has_reason = F.col("cancellation_reason").isNotNull()
    has_closed_ts = F.col("closed_ts").isNotNull()
    finished = has_closed_ts | F.col("_f_TS_CLOSED_UNRECOVERABLE")
    has_hours = F.col("labor_hours").isNotNull()
    has_money = (
        F.col("labor_cost").isNotNull()
        & F.col("parts_cost").isNotNull()
        & (has_hours | F.col("_f_L3"))
    )
    no_money = ~(
        has_hours | F.col("labor_cost").isNotNull() | F.col("parts_cost").isNotNull()
    )
    no_downtime = _downtime_set_count() == 0
    is_valid = (
        F.when(
            status == "Open",
            ~started & ~finished & no_money & no_downtime & ~has_reason,
        )
        # In Progress: downtime may be none, running or already finished (DT_COMPLETE
        # checks its fields are all-or-none); section 5.3 only listed none/running
        .when(
            status == "In Progress", started & ~has_closed_ts & no_money & ~has_reason
        )
        .when(status == "Closed", started & finished & has_money & ~has_reason)
        .when(
            status == "Cancelled",
            ~started & finished & no_money & no_downtime & has_reason,
        )
        .otherwise(True)
    )  # unknown status is DOMAIN_CODES' job
    return _set_flags(df, {"STATUS_LIFECYCLE": ~is_valid})


def check_rules(df: DataFrame) -> DataFrame:
    """Equipment resolution plus every check that only flags."""
    df = resolve_equipment(df)
    checks = (
        check_keys_and_codes,
        check_consistency,
        check_timestamps,
        check_downtime,
        check_labour_consistency,
        check_status_lifecycle,
    )
    return reduce(lambda acc, check: check(acc), checks, df)


def clean_worker_ord(bronze: DataFrame) -> tuple[DataFrame, DataFrame]:
    """Bronze rows -> cleaned work orders: latest version, fixes applied, one _f_<RULE> per rule.

    Returns:
        (cleaned, dedup_flags): `cleaned` holds the deduplicated rows. `dedup_flags` holds every
        loaded Bronze row with the DUP_EXACT / VERSION_LATEST flags, for the row flow.
    """
    df = adapt_bronze_columns(bronze)
    check_contracts(df)
    flagged = deduplicate(standardise(filter_loaded(df)))
    kept = flagged.filter(~F.col("_f_DUP_EXACT") & ~F.col("_f_VERSION_LATEST"))
    steps = (fix_timestamps, convert_currency, fix_labour, fix_parts, check_rules)
    return reduce(lambda acc, step: step(acc), steps, kept), flagged


# ======== 3. creating_silver_table: counts, error gate, writes ====================================
Stats = dict[
    str, dict[str, int]
]  # source batch (or "ALL") -> row counts and failed counts


def add_row_quarantine(df: DataFrame) -> DataFrame:
    """_is_row_quarantined: the row never reaches Silver (0 expected today)."""
    flags = [F.col(f"_f_{rule}") for rule in ROW_QUARANTINE_RULES]
    return df.withColumn("_is_row_quarantined", reduce(or_, flags))


def collect_stats(cleaned: DataFrame, dedup_flags: DataFrame) -> Stats:
    """Row counts and failed counts per rule, per source batch plus 'ALL'."""

    def failed(rule: str) -> Column:
        return F.coalesce(F.sum(F.col(f"_f_{rule}").cast("int")), F.lit(0)).alias(rule)

    all_label = F.coalesce(
        "batch_id", F.lit("ALL")
    )  # rollup marks the total row with null
    quarantined = F.col("_is_row_quarantined").cast("int")
    after = (
        cleaned.rollup("batch_id")
        .agg(
            F.count("*").alias("after_dedup"),
            F.coalesce(F.sum(quarantined), F.lit(0)).alias("row_quarantine"),
            *[failed(r) for r in RULES if r not in DEDUP_RULES],
        )
        .withColumn("batch_id", all_label)
    )
    before = (
        dedup_flags.rollup("batch_id")
        .agg(F.count("*").alias("bronze"), *[failed(r) for r in DEDUP_RULES])
        .withColumn("batch_id", all_label)
    )
    rows = after.join(before, "batch_id", "outer").fillna(0).collect()
    return {row["batch_id"]: row.asDict() for row in rows}


def write_dq_results(
    spark: SparkSession, stats: Stats, table: str, run_id: str
) -> None:
    """Append a summary row to the shared dq_results, matching the existing table schema."""
    total = stats["ALL"]
    total_records = total["bronze"]
    valid = total["after_dedup"] - total["row_quarantine"]
    quarantined = total["row_quarantine"]
    spark.createDataFrame(
        [(total_records, valid, quarantined, datetime.now(timezone.utc), "work_orders")],
        schema="total_records_processed LONG, valid_records LONG, quarantined_records LONG, pipeline_run_ts TIMESTAMP, dataset_name STRING",
    ).write.mode("append").saveAsTable(table)


def raise_on_errors(stats: Stats) -> None:
    """Error gate: any ERROR rule above 0 stops the run before Silver or quarantine is touched."""
    firing = {
        r: stats["ALL"][r]
        for r, (_, severity, _) in RULES.items()
        if severity == "ERROR" and stats["ALL"][r] > 0
    }
    if firing:
        detail = ", ".join(f"{rule}={count:,}" for rule, count in firing.items())
        raise DataQualityError(
            f"Error rules fired, Silver and quarantine untouched: {detail}"
        )


def build_silver(df: DataFrame, run_id: str) -> DataFrame:
    """Rows that survive, cast to SILVER_SCHEMA, with run metadata and the row hash."""
    kept = (
        df.filter(~F.col("_is_row_quarantined"))
        .withColumn("_silver_run_id", F.lit(run_id))
        .withColumn("_silver_processed_ts", F.current_timestamp())
    )
    typed = kept.select(
        *[
            F.col(name).cast(dtype).alias(name)
            for name, dtype, _ in SILVER_SCHEMA
            if name != "_silver_row_hash"
        ]
    )
    return typed.withColumn("_silver_row_hash", _row_hash(ROW_HASH_COLS)).select(
        *[name for name, _, _ in SILVER_SCHEMA]
    )


def build_quarantine(df: DataFrame, run_id: str) -> DataFrame:
    """One OPEN record per flagged row and field. Originals stay in Bronze (join on record_hash)."""

    def record(rule: str, kind: str, field: str | None, original: Column) -> DataFrame:
        return df.filter(F.col(f"_f_{rule}")).select(
            F.lit(QUARANTINE_ENTITY).alias("entity"),
            F.coalesce("work_order_id", "_record_hash").alias("record_id"),
            F.lit(rule).alias("rule_id"),
            F.lit(kind).alias("quarantine_type"),
            F.lit(field).cast("string").alias("field_name"),
            F.lit(RULES[rule][2]).alias("reason"),
            original.alias("original_values"),
            F.col("record_version"),
            F.col("_batch_id"),
            F.col("_source_file"),
            F.current_timestamp().alias("quarantine_ts"),
            F.current_timestamp().alias("last_seen_ts"),
            F.lit(run_id).alias("last_run_id"),
            F.lit("OPEN").alias("resolution_status"),
            F.lit(None).cast("timestamp").alias("resolved_ts"),
        )

    parts = []
    for rule, (kind, fields) in QUARANTINE_RULES.items():
        if kind == "ROW":
            parts.append(record(rule, kind, None, F.to_json(F.struct(*BUSINESS_COLS))))
        for field, source in fields:
            value = F.col(source).alias(source.removeprefix("_orig_"))
            parts.append(record(rule, kind, field, F.to_json(F.struct(value))))
    return reduce(DataFrame.unionByName, parts)


def create_tables(spark: SparkSession, tables: dict[str, str]) -> None:
    """Create Silver (with column comments, change data feed) and quarantine when missing."""
    columns = ", ".join(f"{n} {t} COMMENT '{c}'" for n, t, c in SILVER_SCHEMA)
    silver = tables["silver"]
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {silver} ({columns}) USING DELTA "
        "TBLPROPERTIES (delta.enableChangeDataFeed = true)"
    )
    spark.sql(QUARANTINE_DDL.format(table=tables["quarantine"]))


def merge_silver(spark: SparkSession, silver: DataFrame, target: str) -> None:
    """Idempotent MERGE on work_order_id: a rerun on unchanged data changes 0 rows."""
    silver.createOrReplaceTempView("_silver_updates")
    result = spark.sql(f"""
        MERGE INTO {target} t USING _silver_updates s ON t.work_order_id = s.work_order_id
        WHEN MATCHED AND s.record_version > t.record_version THEN UPDATE SET *
        WHEN MATCHED AND s.record_version = t.record_version
             AND s._silver_row_hash <> t._silver_row_hash THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        WHEN NOT MATCHED BY SOURCE THEN DELETE""").first()
    print(
        f"Silver MERGE: inserted {result['num_inserted_rows']:,}, "
        f"updated {result['num_updated_rows']:,}, deleted {result['num_deleted_rows']:,}"
    )


def merge_quarantine(spark: SparkSession, quarantine: DataFrame, target: str) -> None:
    """MERGE, never append. Key includes field_name (FX_UNAVAILABLE writes one per field)."""
    quarantine.createOrReplaceTempView("_quarantine_updates")
    spark.sql(f"""
        MERGE INTO {target} t USING _quarantine_updates s
        ON t.entity = s.entity AND t.record_id = s.record_id
           AND t.rule_id = s.rule_id AND t.field_name <=> s.field_name
        WHEN MATCHED THEN UPDATE SET last_seen_ts = s.last_seen_ts, last_run_id = s.last_run_id,
             resolution_status = 'OPEN', resolved_ts = NULL
        WHEN NOT MATCHED THEN INSERT *
        WHEN NOT MATCHED BY SOURCE AND t.entity = '{QUARANTINE_ENTITY}'
             AND t.resolution_status = 'OPEN'
             THEN UPDATE SET resolution_status = 'RESOLVED', resolved_ts = current_timestamp()""")


def print_report(stats: Stats) -> None:
    """Print the row flow (every Bronze row has one outcome) and the rules that fired."""
    batches = sorted(b for b in stats if b != "ALL") + ["ALL"]
    flow = [
        ("Bronze (loaded)", "bronze"),
        ("- exact duplicates", "DUP_EXACT"),
        ("- superseded versions", "VERSION_LATEST"),
        ("= after dedup", "after_dedup"),
        ("- row quarantine", "row_quarantine"),
    ]

    def show(label: str, values: list[int], extra: str = "") -> None:
        print(f"{label:<34}{extra:<18}" + "".join(f"{v:>21,}" for v in values))

    print(f"\n{'Row flow':<52}" + "".join(f"{b:>21}" for b in batches))
    for label, key in flow:
        show(label, [stats[b][key] for b in batches])
    show(
        "= Silver",
        [stats[b]["after_dedup"] - stats[b]["row_quarantine"] for b in batches],
    )
    print(f"\n{'Rules that fired':<52}")
    for rule, (_, severity, _) in RULES.items():
        if stats["ALL"][rule]:
            show(rule, [stats[b][rule] for b in batches], severity)


def creating_silver_table(
    spark: SparkSession,
    cleaned: DataFrame,
    dedup_flags: DataFrame,
    tables: dict[str, str],
    run_id: str,
) -> None:
    """Count rules, log them, apply the error gate, then MERGE Silver and quarantine."""
    cleaned = add_row_quarantine(cleaned)
    stats = collect_stats(cleaned, dedup_flags)
    write_dq_results(
        spark, stats, tables["audit"], run_id
    )  # also written when the gate stops
    print_report(stats)
    raise_on_errors(stats)
    create_tables(spark, tables)
    merge_silver(spark, build_silver(cleaned, run_id), tables["silver"])
    merge_quarantine(spark, build_quarantine(cleaned, run_id), tables["quarantine"])


# ======== 4. main =================================================================================

"""Read the widgets, clean Bronze, write Silver, quarantine and DQ results."""
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
dbutils = globals()["dbutils"]  # injected by the notebook runtime
dbutils.widgets.text("catalog", "data2026acc123456")
dbutils.widgets.text("schema", "team3")
catalog, schema = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
if not catalog or not schema:
    raise ValueError("Fill the 'catalog' and 'schema' widgets.")
tables = qualify_tables(catalog, schema)
cleaned, dedup_flags = clean_worker_ord(spark.table(tables["bronze"]))
creating_silver_table(spark, cleaned, dedup_flags, tables, run_id=str(uuid.uuid4()))
print(f"\nDone. FX source: {FX_RATE_SOURCE}.")


Bronze rows read: 20,000; skipped (load_status <> SUCCESS): 0

Row flow                                                BATCH-20260915-01    BATCH-20260922-02                  ALL
Bronze (loaded)                                                    18,679                1,321               20,000
- exact duplicates                                                    189                   11                  200
- superseded versions                                               9,765                  451               10,216
= after dedup                                                       8,725                  859                9,584
- row quarantine                                                        0                    0                    0
= Silver                                                            8,725                  859                9,584

Rules that fired                                    
DUP_EXACT                         DROP                                1

## Silver Sensor Events

In [0]:
"""

Run after silver_equipment, over stable Bronze input. Full refresh outputs.
Shared conventions: Config, SOURCE_COLS/LINEAGE/CASTS, _f_* flags,
clean_bronze_table(), convert_to_silver_table(), main(), UTC, uppercase equipment IDs.
Sensor business key: (source_system, event_id), preserving event/source ID case.
Exact duplicates are compared within a source, excluding ingestion lineage.
All Bronze rows are counted; rescued/non-SUCCESS rows are filtered from Silver.
Missing equipment stays NULL / MISSING in retained rows. Rejected rows get one ROW record
per failed rule in the shared silver_dq_quarantine (same schema as silver_maintenance.py).
Unregistered equipment is retained with dq_unknown_equipment=True.
TYPE_CAST flags identify nonempty values that failed conversion; required sensor
fields are rejected by their DQ rules, optional fields remain NULL with evidence.
No timestamp repair is invented: a sensor event has no maintenance duration.
No fallback to an older valid event when the latest version is invalid.

The supplied silver_maintenance.py is a Work Orders implementation (including its
input/output tables), not a Maintenance History implementation. This script does
not certify or correct that file. It does not need Work Orders or Maintenance.
"""




# ---------------------------------------------------------------- 1. Config & Setup
class Config:
    base = "data2026acc123456.team3"
    # Table names (without the catalog.schema prefix in `base`).
    bronze_table = "test_bronze_sensor_events"
    equipment_table = "test_silver_equipment"
    silver_table = "test_silver_sensor_events"
    superseded_table = "test_superseded_sensor_events"
    quarantine_table = "test_silver_dq_quarantine"
    quarantine_updates_view = "test_quarantine_updates"
    audit_table = "test_dq_results"
    dataset_name = "sensor_events"
    bronze_loaded_status = "SUCCESS"
    bronze_column_map = {
        "ingestion_timestamp": "_ingestion_ts",
        "batch_id": "_batch_id",
        "source_filename": "_source_file",
        "record_hash": "_record_hash",
        "load_status": "_load_status",
    }


CFG = Config()


# ---------------------------------------------------------------- 2. Columns & Expressions
REQUIRED_PATHS = ['event_id',
 'record_version',
 'event_ts',
 'equipment_id',
 'operating_state',
 'sensor.sensor_id',
 'sensor.sensor_type',
 'sensor.measure',
 'sensor.canonical_unit',
 'sensor.range_min',
 'sensor.range_max',
 'sensor.measurement_point',
 'reading.value',
 'reading.unit',
 'quality_flag',
 'alarm.state',
 'alarm.code',
 'alarm.limit_value',
 'gateway.gateway_id',
 'gateway.firmware_version',
 'gateway.received_ts',
 'location.site_code',
 'location.production_line',
 'tags',
 '_metadata.source_system',
 '_metadata.source_system_key',
 '_metadata.source_file',
 '_metadata.batch_id',
 '_metadata.ingestion_ts',
 'record_hash',
 'ingestion_timestamp',
 'source_filename',
 'batch_id',
 'load_status']
SOURCE_COLS = ['event_id',
 'record_version',
 'event_timestamp',
 'equipment_id',
 'operating_state',
 'sensor_id',
 'sensor_type',
 'measure',
 'canonical_unit',
 'range_min',
 'range_max',
 'measurement_point',
 'reading_value_raw',
 'reading_value',
 'reading_unit',
 'quality_flag',
 'alarm_state',
 'alarm_code',
 'alarm_limit_value',
 'gateway_id',
 'firmware_version',
 'received_timestamp',
 'site_code',
 'production_line',
 'tags',
 'source_system',
 'source_system_key',
 'source_file',
 'source_batch_id',
 'source_ingestion_timestamp']
CASTS = {'record_version': ('record_version', 'INT'),
 'event_timestamp': ('event_ts', 'TIMESTAMP'),
 'range_min': ('sensor.range_min', 'DOUBLE'),
 'range_max': ('sensor.range_max', 'DOUBLE'),
 'reading_value': ('reading.value', 'DOUBLE'),
 'alarm_limit_value': ('alarm.limit_value', 'DOUBLE'),
 'received_timestamp': ('gateway.received_ts', 'TIMESTAMP'),
 'source_ingestion_timestamp': ('_metadata.ingestion_ts', 'TIMESTAMP'),
 'bronze_ingestion_timestamp': ('ingestion_timestamp', 'TIMESTAMP')}
LINEAGE = ["source_system", "source_system_key", "source_file",
           "source_batch_id", "source_ingestion_timestamp"]
BUSINESS = [c for c in SOURCE_COLS if c not in LINEAGE]


def B(expression):
    return F.coalesce(expression, F.lit(False))


def FLAG(name):
    return f"_f_{name}"


def fired(pairs):
    return F.filter(F.array(*[F.when(B(condition), F.lit(name))
                             for name, condition in pairs]),
                    lambda value: value.isNotNull())


# Shared silver_dq_quarantine schema: identical in silver_maintenance.py and silver-stau.py
# (spec: context/silver_max/silver_maintenance_rules.md 6.3). Change all three together.
QUARANTINE_DDL = """CREATE TABLE IF NOT EXISTS {table} (
    entity STRING, record_id STRING, rule_id STRING, quarantine_type STRING, field_name STRING,
    reason STRING, original_values STRING, record_version INT, _batch_id STRING,
    _source_file STRING, quarantine_ts TIMESTAMP, last_seen_ts TIMESTAMP, last_run_id STRING,
    resolution_status STRING, resolved_ts TIMESTAMP
) USING DELTA"""
QUARANTINE_ENTITY = "test_sensor_events"


@dataclass
class CleaningResult:
    silver: DataFrame
    superseded: DataFrame
    quarantine: DataFrame
    counts: dict
    run_id: str


# ---------------------------------------------------------------- 3. Core Functions

def _check_schema(df: DataFrame, paths, label: str) -> None:
    missing = []
    for path in paths:
        current = df.schema
        for part in path.split("."):
            if not isinstance(current, StructType) or part not in current.fieldNames():
                missing.append(path)
                break
            current = current[part].dataType
    if missing:
        raise ValueError(f"SCHEMA_CONTRACT: {label} missing/invalid paths: {missing}")


def _read_and_standardise(bronze: DataFrame) -> DataFrame:
    _check_schema(bronze, REQUIRED_PATHS, "bronze_sensor_events")
    if "_rescued_data" not in bronze.columns:
        bronze = bronze.withColumn("_rescued_data", F.lit(None).cast("string"))
    # Snapshot cast failures before standardization loses the original values.
    cast_errors = []
    for name, (path, dtype) in CASTS.items():
        quoted = ".".join(f"`{p}`" for p in path.split("."))
        raw = F.trim(F.col(quoted).cast("string"))
        cast_errors.append((name, raw.isNotNull() & (raw != "") &
                            F.expr(f"try_cast({quoted} AS {dtype})").isNull()))
    bronze = bronze.withColumn("_type_cast_fields", fired(cast_errors))
    df = bronze.selectExpr(
        # Event identifiers and timestamps
        "trim(event_id) AS event_id",
        "try_cast(record_version AS INT) AS record_version",
        "try_cast(event_ts AS TIMESTAMP) AS event_timestamp",
        "nullif(trim(equipment_id), '') AS equipment_id",
        "lower(trim(operating_state)) AS operating_state",

        # Sensor details
        "trim(sensor.sensor_id) AS sensor_id",
        "lower(trim(sensor.sensor_type)) AS sensor_type",
        "trim(sensor.measure) AS measure",
        "trim(sensor.canonical_unit) AS canonical_unit",
        "try_cast(sensor.range_min AS DOUBLE) AS range_min",
        "try_cast(sensor.range_max AS DOUBLE) AS range_max",
        "trim(sensor.measurement_point) AS measurement_point",

        # Reading: preserve original value and unit
        "reading.value AS reading_value_raw",
        "try_cast(reading.value AS DOUBLE) AS reading_value",
        "trim(reading.unit) AS reading_unit",

        # Quality and alarm details
        "upper(trim(quality_flag)) AS quality_flag",
        "lower(trim(alarm.state)) AS alarm_state",
        "trim(alarm.code) AS alarm_code",
        "try_cast(alarm.limit_value AS DOUBLE) AS alarm_limit_value",

        # Gateway and location
        "trim(gateway.gateway_id) AS gateway_id",
        "gateway.firmware_version AS firmware_version",
        "try_cast(gateway.received_ts AS TIMESTAMP) AS received_timestamp",
        "upper(trim(location.site_code)) AS site_code",
        "trim(location.production_line) AS production_line",
        "tags",

        # Source metadata
        "nullif(trim(`_metadata`.source_system), '') AS source_system",
        "`_metadata`.source_system_key AS source_system_key",
        "`_metadata`.source_file AS source_file",
        "`_metadata`.batch_id AS source_batch_id",
        "try_cast(`_metadata`.ingestion_ts AS TIMESTAMP) AS source_ingestion_timestamp",

        # Bronze metadata for traceability
        "record_hash",
        "try_cast(ingestion_timestamp AS TIMESTAMP) AS bronze_ingestion_timestamp",
        "source_filename",
        "batch_id AS bronze_batch_id",
        "load_status",
        "_rescued_data",

        # Original record for investigation and quarantine
        "to_json(struct(*)) AS raw_record_json",
        "_type_cast_fields"
    )

    # All projected string fields: trim and empty -> NULL. Keep raw evidence intact.
    df = df.select(*[
        F.when(F.trim(F.col(name)) == "", None).otherwise(F.trim(F.col(name))).alias(name)
        if dtype == "string" and name not in ["raw_record_json", "_rescued_data"]
        else F.col(name)
        for name, dtype in df.dtypes
    ])
    return (df.withColumn("equipment_id", F.upper("equipment_id"))
            .withColumn(FLAG("TYPE_CAST"), F.size("_type_cast_fields") > 0))


def _deduplicate(df: DataFrame) -> DataFrame:
    # Include invalid cast originals so different malformed inputs do not collapse
    # into an identical NULL business payload after casting.
    invalid_originals = F.struct(*[
        F.when(F.array_contains("_type_cast_fields", name),
               F.get_json_object("raw_record_json", "$." + path)).alias(name)
        for name, (path, _) in CASTS.items()
        if name in BUSINESS
    ])
    business_json = F.to_json(F.struct(
        *[F.col(name) for name in BUSINESS], invalid_originals.alias("cast_originals")),
        options={"ignoreNullFields": "false"})
    df = df.withColumn("_business_json", business_json)
    eligible = (F.col("event_id").isNotNull() & F.col("source_system").isNotNull()
                & B(F.col("load_status") == CFG.bronze_loaded_status)
                & F.col("_rescued_data").isNull())
    # Do not let malformed/rescued rows displace a successfully ingested event.
    df = df.withColumn("_dedup_eligible", eligible)
    w1 = Window.partitionBy("source_system", "_business_json", "_dedup_eligible").orderBy(
        F.col("bronze_ingestion_timestamp").asc_nulls_last(),
        F.col("source_filename").asc_nulls_last(),
        F.col("record_hash").asc_nulls_last(), "raw_record_json")
    df = df.withColumn(FLAG("DUP_EXACT"), eligible & (F.row_number().over(w1) > 1))
    w2 = Window.partitionBy("source_system", "event_id", "_dedup_eligible",
                            FLAG("DUP_EXACT")).orderBy(
        F.col("record_version").desc_nulls_last(),
        F.col("source_ingestion_timestamp").desc_nulls_last(),
        F.col("bronze_ingestion_timestamp").desc_nulls_last(),
        F.col("record_hash").asc_nulls_last(), "raw_record_json")
    return (df.withColumn(FLAG("VERSION_LATEST"), eligible &
                         ~F.col(FLAG("DUP_EXACT")) & (F.row_number().over(w2) > 1))
            .drop("_business_json", "_dedup_eligible"))


def clean_bronze_table(bronze: DataFrame, equipment: DataFrame) -> CleaningResult:
    """Standardize, deduplicate, validate and route; do not write tables."""
    bronze.sparkSession.conf.set("spark.sql.session.timeZone", "UTC")
    RUN_ID = f"se-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}-{uuid.uuid4().hex[:6]}"
    _check_schema(equipment, ["equipment_id"], "silver_equipment")
    if bronze.limit(1).count() == 0:
        raise ValueError("Empty Bronze: stopped before replacing Silver outputs")
    raw = _deduplicate(_read_and_standardise(bronze))
    excluded = F.col(FLAG("DUP_EXACT")) | F.col(FLAG("VERSION_LATEST"))
    df_superseded = (raw.filter(excluded).withColumn("superseded_reason",
        F.when(F.col(FLAG("DUP_EXACT")), "DUP_EXACT").otherwise("VERSION_LATEST")))
    df_cleaned = raw.filter(~excluded)
    input_count = raw.count()
    superseded_count = df_superseded.count()
    deduped_count = df_cleaned.count()
    assert input_count == superseded_count + deduped_count
    df_normalized = df_cleaned.withColumn(
        "reading_value_canonical",

        # Already in the correct unit
        F.when(
            F.col("reading_unit") == F.col("canonical_unit"),
            F.col("reading_value")
        )

        # Fahrenheit -> Celsius
        .when(
            (F.col("reading_unit") == "degF") &
            (F.col("canonical_unit") == "degC"),
            (F.col("reading_value") - 32.0) * 5.0 / 9.0
        )

        # Inches/second -> Millimeters/second
        .when(
            (F.col("reading_unit") == "in/s") &
            (F.col("canonical_unit") == "mm/s"),
            F.col("reading_value") * 25.4
        )

        # PSI -> Bar
        .when(
            (F.col("reading_unit") == "psi") &
            (F.col("canonical_unit") == "bar"),
            F.col("reading_value") * 0.0689475729
        )

        # Unsupported or missing unit: flag later in quality checks
        .otherwise(F.lit(None).cast("double"))
    )

    # Read only the equipment IDs. No changes to silver_equipment.
    equipment_keys = (
        equipment
        .select(F.upper(F.trim(F.col("equipment_id"))).alias("equipment_id"))
        .filter(
            F.col("equipment_id").isNotNull() &
            (F.col("equipment_id") != "")
        )
        .distinct()
        .withColumn("equipment_exists", F.lit(True))
    )

    # Keep every sensor event, including unmatched equipment IDs.
    df_checked = (
        df_normalized
        .join(equipment_keys, on="equipment_id", how="left")
        .withColumn(
            "equipment_exists",
            F.coalesce(F.col("equipment_exists"), F.lit(False))
        )
    )

    # Shared equipment flags; equipment ID case is normalized to uppercase.
    # No work-order-based ID recovery is available in the sensor pipeline.
    df_checked = (df_checked
        .withColumn("dq_unknown_equipment",
                    F.col("equipment_id").isNotNull() & ~F.col("equipment_exists"))
        .withColumn("equipment_resolution",
                    F.when(F.col("equipment_id").isNull(), F.lit("MISSING"))
                     .when(F.col("dq_unknown_equipment"), F.lit("UNREGISTERED"))
                     .otherwise(F.lit("SOURCE"))))
    df_checked = (df_checked
        .withColumn(FLAG("EQ_MISSING"), F.col("equipment_id").isNull())
        .withColumn(FLAG("EQ_UNREGISTERED"), F.col("dq_unknown_equipment")))
    # Missing IDs are retained; other sensor DQ errors filter out the row.

    df_deduped = df_checked
    def missing_text(column_name):
        return (
            F.col(column_name).isNull() |
            (F.trim(F.col(column_name)) == "")
        )


    def invalid_number(column_name):
        return (
            F.col(column_name).isNull() |
            F.isnan(F.col(column_name)) |
            (F.abs(F.col(column_name)) == F.lit(float("inf")))
        )


    # Each condition describes a quality failure.
    rules = [
        (
            "DQ01: Missing event_id",
            missing_text("event_id")
        ),
        (
            "DQ02: Missing source_system",
            missing_text("source_system")
        ),
        # DQ03 retired: missing equipment is EQ_MISSING (FLAG only).
        # DQ04 retired: unknown equipment is now EQ_UNREGISTERED (FLAG), not an error.
        (
            "DQ05: Missing sensor_id",
            missing_text("sensor_id")
        ),
        (
            "DQ06: Missing or invalid event timestamp",
            F.col("event_timestamp").isNull()
        ),
        (
            "DQ07: Missing or invalid record_version",
            F.col("record_version").isNull() |
            (F.col("record_version") < 1)
        ),
        (
            "DQ08: Missing, invalid reading or unsupported unit conversion",
            invalid_number("reading_value_canonical")
        ),
        (
            "DQ09: Missing or invalid sensor range",
            invalid_number("range_min") |
            invalid_number("range_max") |
            (F.col("range_min") > F.col("range_max"))
        ),
        (
            "DQ10: Reading outside sensor range",
            (F.col("reading_value_canonical") < F.col("range_min")) |
            (F.col("reading_value_canonical") > F.col("range_max"))
        ),
        (
            "DQ11: Missing or invalid operating_state",
            F.col("operating_state").isNull() |
            (~F.col("operating_state").isin(
                "running", "stopped", "maintenance"
            ))
        ),
        (
            "DQ12: Missing or invalid alarm_state",
            F.col("alarm_state").isNull() |
            (~F.col("alarm_state").isin(
                "normal", "warning", "alarm"
            ))
        ),
        (
            "DQ13: Rescued or non-SUCCESS source data requires review",
            F.col("_rescued_data").isNotNull() |
            ~B(F.col("load_status") == CFG.bronze_loaded_status)
        )
    ]

    # Build an array containing every failed rule.
    df_evaluated = (
        df_deduped
        .withColumn(
            "dq_errors",
            F.filter(
                F.array(*[
                    F.when(
                        F.coalesce(condition, F.lit(False)),
                        F.lit(reason)
                    )
                    for reason, condition in rules
                ]),
                lambda reason: reason.isNotNull()
            )
        )
        .withColumn(
            "is_valid",
            F.size(F.col("dq_errors")) == 0
        )
    )

    # Valid sensor events.
    # Retain quality_flag and alarm_state for downstream analysis.
    df_valid = (
        df_evaluated
        .filter(F.col("is_valid"))
        .drop("is_valid", "dq_errors")
        .withColumn("silver_processed_ts", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(RUN_ID))
    )

    # One ROW record per rejected row and failed rule: "DQ10: Reading outside..." -> rule + reason.
    failure = F.split(F.col("failure"), ": ", 2)
    now = F.current_timestamp()
    df_quarantine = (df_evaluated.filter(~F.col("is_valid"))
        .select(F.explode("dq_errors").alias("failure"), "event_id", "record_hash",
                "record_version", "bronze_batch_id", "source_filename", "raw_record_json")
        .select(
            F.lit(QUARANTINE_ENTITY).alias("entity"),
            F.coalesce("event_id", "record_hash").alias("record_id"),
            failure[0].alias("rule_id"), F.lit("ROW").alias("quarantine_type"),
            F.lit(None).cast("string").alias("field_name"), failure[1].alias("reason"),
            F.col("raw_record_json").alias("original_values"), "record_version",
            F.col("bronze_batch_id").alias("_batch_id"),
            F.col("source_filename").alias("_source_file"),
            now.alias("quarantine_ts"), now.alias("last_seen_ts"),
            F.lit(RUN_ID).alias("last_run_id"), F.lit("OPEN").alias("resolution_status"),
            F.lit(None).cast("timestamp").alias("resolved_ts")))

    valid_count = df_valid.count()
    rejected_count = df_evaluated.filter(~F.col("is_valid")).count()
    missing_equipment_count = df_valid.filter(F.col("equipment_id").isNull()).count()
    assert deduped_count == valid_count + rejected_count, "DQ row reconciliation failed"
    print(f"Silver: {valid_count}; filtered invalid: {rejected_count}; "
          f"retained missing equipment: {missing_equipment_count}")
    return CleaningResult(
        silver=df_valid, superseded=df_superseded, quarantine=df_quarantine,
        run_id=RUN_ID,
        counts={"input": input_count, "silver": valid_count,
                "rejected": rejected_count,
                "missing_equipment_retained": missing_equipment_count,
                "superseded": superseded_count})


def _with_silver_metadata(df: DataFrame, run_id: str) -> DataFrame:
    # Keep sensor column names; add the common lineage names used by Work Orders.
    aliases = {
        "source_ingestion_timestamp": "ingestion_ts", "source_batch_id": "batch_id",
        "bronze_ingestion_timestamp": "_ingestion_ts", "bronze_batch_id": "_batch_id",
        "source_filename": "_source_file", "record_hash": "_record_hash",
        "source_system": "_source_system", "load_status": "_load_status",
    }
    for source, target in aliases.items():
        df = df.withColumn(target, F.col(source))
    hash_cols = BUSINESS + ["source_system", "reading_value_canonical",
                            "equipment_resolution", "dq_unknown_equipment"]
    # JSON avoids separator/null-token collisions in concat-based row hashes.
    return (df.withColumn("_silver_row_hash", F.sha2(F.to_json(
                F.struct(*[F.col(c) for c in hash_cols]),
                options={"ignoreNullFields": "false"}), 256))
            .withColumn("_silver_run_id", F.lit(run_id))
            .withColumn("_silver_processed_ts", F.current_timestamp())
            .drop("silver_run_id", "silver_processed_ts"))


def merge_quarantine(spark, quarantine: DataFrame, target: str) -> None:
    """MERGE, never append: reruns refresh last_seen_ts, records that stop firing turn RESOLVED."""
    spark.sql(QUARANTINE_DDL.format(table=target))
    # Rows without event_id share record_hash when Bronze holds identical copies.
    quarantine.dropDuplicates(["record_id", "rule_id"]).createOrReplaceTempView(
        CFG.quarantine_updates_view)
    spark.sql(f"""
        MERGE INTO {target} t USING {CFG.quarantine_updates_view} s
        ON t.entity = s.entity AND t.record_id = s.record_id
           AND t.rule_id = s.rule_id AND t.field_name <=> s.field_name
        WHEN MATCHED THEN UPDATE SET last_seen_ts = s.last_seen_ts, last_run_id = s.last_run_id,
             resolution_status = 'OPEN', resolved_ts = NULL
        WHEN NOT MATCHED THEN INSERT *
        WHEN NOT MATCHED BY SOURCE AND t.entity = '{QUARANTINE_ENTITY}'
             AND t.resolution_status = 'OPEN'
             THEN UPDATE SET resolution_status = 'RESOLVED', resolved_ts = current_timestamp()""")


def convert_to_silver_table(cleaned: CleaningResult, base: str = CFG.base) -> None:
    """Validate row reconciliation, write entity outputs, append shared audit."""
    spark = cleaned.silver.sparkSession
    counts = cleaned.counts
    assert counts["input"] == (counts["silver"] + counts["rejected"]
                                + counts["superseded"]), "Row reconciliation failed"
    assert cleaned.silver.groupBy("source_system", "event_id").count().filter(
        F.col("count") > 1).limit(1).count() == 0, "Duplicate Silver event keys"
    assert cleaned.silver.filter(F.col("equipment_id").isNull() & (
        ~F.col("equipment_resolution").eqNullSafe("MISSING") |
        ~F.col("dq_unknown_equipment").eqNullSafe(False)
    )).limit(1).count() == 0, "Inconsistent missing-equipment flags"
    assert cleaned.silver.filter(F.col("equipment_id").isNull()).count() == counts["missing_equipment_retained"]

    audit_table = f"{base}.{CFG.audit_table}"
    audit_df = spark.createDataFrame(
        [(counts["input"], counts["silver"], counts["rejected"],
          datetime.now(timezone.utc), CFG.dataset_name)],
        schema="total_records_processed LONG, valid_records LONG, quarantined_records LONG, "
               "pipeline_run_ts TIMESTAMP, dataset_name STRING",
    )
    if spark.catalog.tableExists(audit_table):
        existing = {f.name: f.dataType for f in spark.table(audit_table).schema}
        incompatible = [f.name for f in audit_df.schema
                        if f.name in existing and existing[f.name] != f.dataType]
        if incompatible:
            raise ValueError(f"Incompatible dq_results column types: {incompatible}")

    # Full refresh; each Delta write is atomic, not the group of writes.
    for name, frame, expected in [
        (CFG.silver_table, cleaned.silver, counts["silver"]),
        (CFG.superseded_table, cleaned.superseded, counts["superseded"]),
    ]:
        final = _with_silver_metadata(frame, cleaned.run_id) if name == CFG.silver_table else (
            frame.withColumn("_silver_run_id", F.lit(cleaned.run_id))
                 .withColumn("_silver_processed_ts", F.current_timestamp())
                 .drop("silver_run_id"))
        table = f"{base}.{name}"
        final.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true").saveAsTable(table)
        assert spark.table(table).count() == expected, f"Saved row count mismatch: {table}"
        print(f"{table}: {expected} records saved")
    merge_quarantine(spark, cleaned.quarantine, f"{base}.{CFG.quarantine_table}")
    audit_df.write.format("delta").mode("append").saveAsTable(audit_table)
    print(f"Run {cleaned.run_id}: {counts}")
    print("Missing-equipment rows are included in Silver; invalid rows are in silver_dq_quarantine.")


# ---------------------------------------------------------------- 4. Main Execution


"""Run in Databricks with an active Spark session."""
spark = SparkSession.getActiveSession()
if spark is None:
    raise RuntimeError("Run this script in Databricks with an active Spark session")
cleaned = clean_bronze_table(
    bronze=spark.table(f"{CFG.base}.{CFG.bronze_table}"),
    equipment=spark.table(f"{CFG.base}.{CFG.equipment_table}"))
convert_to_silver_table(cleaned)

Silver: 9797; filtered invalid: 73; retained missing equipment: 51
data2026acc123456.team3.test_silver_sensor_events: 9797 records saved
data2026acc123456.team3.test_superseded_sensor_events: 10130 records saved
Run se-20261008T121307-bbd2ca: {'input': 20000, 'silver': 9797, 'rejected': 73, 'missing_equipment_retained': 51, 'superseded': 10130}
Missing-equipment rows are included in Silver; invalid rows are in silver_dq_quarantine.


# GOLD TRANSFORMATION

New good gold

In [0]:
# ==============================================================================
# GOLD - CELL 1 of 3: shared setup + DIMENSION tables
# Run order: this cell -> facts cell -> KPI cell
# (facts need the dimension keys, so the dimensions come first)
# ==============================================================================

# ---------------------------------------------------------------- 1. Table names
# Change a table name here and the whole Gold layer follows.
BASE = f"{CATALOG_NAME}.{SCHEMA_NAME}"

SILVER_EQUIPMENT = f"{BASE}.test_silver_equipment"
SILVER_WORK_ORDERS = f"{BASE}.test_silver_work_orders"
SILVER_MAINTENANCE_HISTORY = f"{BASE}.test_silver_maintenance_history"
SILVER_SENSOR_EVENTS = f"{BASE}.test_silver_sensor_events"

DIM_DATE = f"{BASE}.test_gold_dim_date"
DIM_SITE = f"{BASE}.test_gold_dim_site"
DIM_EQUIPMENT_TYPE = f"{BASE}.test_gold_dim_equipment_type"
DIM_EQUIPMENT = f"{BASE}.test_gold_dim_equipment"
DIM_FAILURE_MODE = f"{BASE}.test_gold_dim_failure_mode"
DIM_MAINTENANCE_TYPE = f"{BASE}.test_gold_dim_maintenance_type"

FACT_SENSOR_READING = f"{BASE}.test_gold_fact_sensor_reading"
FACT_WORK_ORDER = f"{BASE}.test_gold_fact_maintenance_work_order"
FACT_DOWNTIME = f"{BASE}.test_gold_fact_downtime"

KPI_TABLE = f"{BASE}.test_gold_kpi_equipment_month"
KPI_VIEW = f"{BASE}.test_gold_vw_kpi_equipment_month_v2"

# ---------------------------------------------------------------- 2. Settings
UNKNOWN_KEY = -1                 # key of the "Unknown" row in every dimension
AS_OF_OVERRIDE = None            # e.g. "2026-09-30 23:59:59"; None = latest timestamp in Silver
CLOSED_STATUSES = ["Closed", "Cancelled"]
CANCELLED_STATUSES = ["Cancelled"]  # cancelled work orders: kept as rows, excluded from counts and hours_to_close
UNPLANNED_TYPES = ["Corrective"]  # only Corrective work orders are unplanned
ALARM_STATES = ["warning", "alarm"]
TEMP_FIX = "Temporary fix"
REPEAT_DAYS = 30                 # repeat failure = same asset + failure mode within 30 days
TS_FORMAT = "yyyy-MM-dd HH:mm:ss"

FAILURE_MODES = {
    "NONE": "No failure (planned work)",
    "BRG-WEAR": "Bearing wear",
    "MISALIGN": "Shaft misalignment",
    "IMBAL": "Rotor imbalance",
    "OVERHEAT": "Overheating",
    "LUB-FAIL": "Lubrication failure",
    "WIND-INS": "Stator winding insulation breakdown",
    "SEAL-LEAK": "Mechanical seal leakage",
    "CAVIT": "Cavitation",
    "VALVE-FAIL": "Valve failure",
    "GEAR-WEAR": "Gear tooth wear",
    "CLOG": "Blocked suction / plugged strainer",
}
MAINTENANCE_TYPES = ["Preventive", "Corrective", "Inspection", "Predictive"]

spark.conf.set("spark.sql.session.timeZone", "UTC")


# ---------------------------------------------------------------- 3. Small helpers
# (the facts and KPI cells reuse these)
def clean_code(column):
    """Trim, upper-case, empty -> NULL. Used for equipment IDs and site codes."""
    return F.nullif(F.upper(F.trim(F.col(column))), F.lit(""))


def make_key(column):
    """Surrogate key: a hash of the business key. Same input = same key on every run."""
    return F.abs(F.xxhash64(column))


def key_or_unknown(column):
    return F.when(column.isNull(), F.lit(UNKNOWN_KEY)).otherwise(make_key(column))


def date_key(column):
    """Timestamp or date -> 20260315. NULL -> -1 (Unknown)."""
    return F.when(column.isNull(), F.lit(UNKNOWN_KEY)).otherwise(
        F.date_format(column, "yyyyMMdd").cast("int")
    )


def hours_between(start, end):
    return (F.unix_timestamp(end) - F.unix_timestamp(start)) / 3600.0


def add_unknown_row(df, **values):
    """Add the Unknown member (key -1). Columns that are not given stay NULL."""
    row = spark.range(1).select(
        *[F.lit(values.get(f.name)).cast(f.dataType).alias(f.name) for f in df.schema.fields]
    )
    return df.unionByName(row)


def save_table(df, table):
    """Overwrite the Gold table. overwriteSchema is on, so new or changed columns replace the old
    schema without dropping the table. Schema changes no longer fail loudly: rely on the checks."""
    df = df.withColumn("gold_loaded_ts", F.current_timestamp())  # last-refresh indicator
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table}: {spark.table(table).count()} rows")


FAILED_CHECKS = []


def check(name, expected, actual, tolerance=0):
    """Print PASS / FAIL for one reconciliation check."""
    expected, actual = float(expected or 0), float(actual or 0)
    ok = abs(actual - expected) <= tolerance
    print(f"{'PASS' if ok else 'FAIL'} | {name} | expected={expected:,.2f} actual={actual:,.2f}")
    if not ok:
        FAILED_CHECKS.append(name)


FAILED_CHECKS.clear()

# ---------------------------------------------------------------- 4. Read Silver
silver_equipment = spark.table(SILVER_EQUIPMENT)
silver_wo = spark.table(SILVER_WORK_ORDERS)
silver_sensor = spark.table(SILVER_SENSOR_EVENTS)

# As-of timestamp = latest event in Silver. Used later for overdue work orders and
# downtime that has not ended yet. Fixed here so every Gold table uses the same value.
wo_event_cols = ["created_ts", "actual_start_ts", "closed_ts", "downtime_start_ts", "downtime_end_ts"]
latest_sensor = silver_sensor.agg(F.date_format(F.max("event_timestamp"), TS_FORMAT)).first()[0]
latest_wo = silver_wo.agg(F.date_format(F.max(F.greatest(*wo_event_cols)), TS_FORMAT)).first()[0]
AS_OF = AS_OF_OVERRIDE or max(t for t in [latest_sensor, latest_wo] if t)
print("As-of timestamp:", AS_OF)

# Date range of dim_date: whole months from the first to the last date in the data
# (due_ts can be in the future, so it is included)
wo_date_cols = wo_event_cols + ["due_ts"]
first_date = min(
    silver_sensor.agg(F.min(F.col("event_timestamp").cast("date"))).first()[0],
    silver_wo.agg(F.min(F.least(*wo_date_cols)).cast("date")).first()[0],
)
last_date = max(
    silver_sensor.agg(F.max(F.col("event_timestamp").cast("date"))).first()[0],
    silver_wo.agg(F.max(F.greatest(*wo_date_cols)).cast("date")).first()[0],
)

# ---------------------------------------------------------------- 5. dim_date
days = spark.range(1).select(
    F.explode(
        F.sequence(F.trunc(F.lit(first_date), "month"), F.last_day(F.lit(last_date)), F.expr("INTERVAL 1 DAY"))
    ).alias("full_date")
)
dim_date = days.select(
    date_key(F.col("full_date")).alias("date_key"),
    "full_date",
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month_number"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.date_format("full_date", "yyyy-MM").alias("year_month"),
    F.trunc("full_date", "month").alias("month_start_date"),  # the KPI table joins on this
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
)
dim_date = add_unknown_row(
    dim_date, date_key=UNKNOWN_KEY, month_name="Unknown", year_month="UNKNOWN", day_name="Unknown"
)

# ---------------------------------------------------------------- 6. Current equipment
# Equipment IDs differ in casing between Silver tables, so every table uses upper(trim()).
equipment = (
    silver_equipment
    .withColumn("equipment_id", clean_code("equipment_id"))
    .withColumn("site_code", clean_code("site_code"))
    .withColumn("equipment_type", F.nullif(F.trim("equipment_type"), F.lit("")))
    .filter(F.col("equipment_id").isNotNull())
)
# keep one row per equipment: the latest version (this makes dim_equipment SCD type 1)
latest_first = Window.partitionBy("equipment_id").orderBy(
    F.col("record_version").desc(), F.col("last_modified_ts").desc()
)
equipment = equipment.withColumn("rn", F.row_number().over(latest_first)).filter("rn = 1").drop("rn")

# ---------------------------------------------------------------- 7. dim_site
dim_site = (
    equipment.filter(F.col("site_code").isNotNull())
    .groupBy("site_code")
    .agg(
        F.max("site_name").alias("site_name"),
        F.max("country").alias("country"),
        F.max("site_timezone").alias("site_timezone"),
    )
    .withColumn("site_key", make_key(F.col("site_code")))
    .select("site_key", "site_code", "site_name", "country", "site_timezone")
)
dim_site = add_unknown_row(dim_site, site_key=UNKNOWN_KEY, site_code="UNKNOWN", site_name="Unknown site")

# ---------------------------------------------------------------- 8. dim_equipment_type
dim_equipment_type = (
    equipment.filter(F.col("equipment_type").isNotNull())
    .select("equipment_type").distinct()
    .withColumn("equipment_type_key", make_key(F.col("equipment_type")))
    .select("equipment_type_key", "equipment_type")
)
dim_equipment_type = add_unknown_row(
    dim_equipment_type, equipment_type_key=UNKNOWN_KEY, equipment_type="UNKNOWN"
)

# ---------------------------------------------------------------- 9. dim_equipment
# operating_hours_per_day / operating_days_per_week are carried so the KPI cell can build operating_share
dim_equipment = equipment.select(
    make_key(F.col("equipment_id")).alias("equipment_key"),
    "equipment_id", "equipment_name", "manufacturer", "model_number", "production_line",
    "criticality", "status", "install_date", "decommission_date", "rated_power_kw",
    "operating_hours_per_day", "operating_days_per_week",
    key_or_unknown(F.col("site_code")).alias("site_key"),
    key_or_unknown(F.col("equipment_type")).alias("equipment_type_key"),
)
dim_equipment = add_unknown_row(
    dim_equipment, equipment_key=UNKNOWN_KEY, equipment_id="UNKNOWN", equipment_name="Unknown equipment",
    site_key=UNKNOWN_KEY, equipment_type_key=UNKNOWN_KEY,
)

# ---------------------------------------------------------------- 10. dim_failure_mode
dim_failure_mode = spark.createDataFrame(
    list(FAILURE_MODES.items()), "failure_mode_code STRING, failure_mode_desc STRING"
).withColumn("failure_mode_key", make_key(F.col("failure_mode_code")))
dim_failure_mode = dim_failure_mode.select("failure_mode_key", "failure_mode_code", "failure_mode_desc")
dim_failure_mode = add_unknown_row(
    dim_failure_mode, failure_mode_key=UNKNOWN_KEY, failure_mode_code="UNKNOWN",
    failure_mode_desc="Unknown failure mode",
)

# ---------------------------------------------------------------- 11. dim_maintenance_type
# is_planned is False only for Corrective work (Predictive is condition-triggered but scheduled)
dim_maintenance_type = spark.createDataFrame(
    [(t, t not in UNPLANNED_TYPES) for t in MAINTENANCE_TYPES],
    "maintenance_type STRING, is_planned BOOLEAN",
).withColumn("maintenance_type_key", make_key(F.col("maintenance_type")))
dim_maintenance_type = dim_maintenance_type.select("maintenance_type_key", "maintenance_type", "is_planned")
dim_maintenance_type = add_unknown_row(
    dim_maintenance_type, maintenance_type_key=UNKNOWN_KEY, maintenance_type="UNKNOWN"
)

# ---------------------------------------------------------------- 12. Save
save_table(dim_date, DIM_DATE)
save_table(dim_site, DIM_SITE)
save_table(dim_equipment_type, DIM_EQUIPMENT_TYPE)
save_table(dim_equipment, DIM_EQUIPMENT)
save_table(dim_failure_mode, DIM_FAILURE_MODE)
save_table(dim_maintenance_type, DIM_MAINTENANCE_TYPE)

# ---------------------------------------------------------------- 13. Checks
# Every surrogate key must be unique (a hash collision would show up here)
for table, key in [
    (DIM_DATE, "date_key"), (DIM_SITE, "site_key"), (DIM_EQUIPMENT_TYPE, "equipment_type_key"),
    (DIM_EQUIPMENT, "equipment_key"), (DIM_FAILURE_MODE, "failure_mode_key"),
    (DIM_MAINTENANCE_TYPE, "maintenance_type_key"),
]:
    df = spark.table(table)
    check(f"{key} is unique in {table}", df.count(), df.select(key).distinct().count())

silver_equipment_count = silver_equipment.select(clean_code("equipment_id")).dropna().distinct().count()
check("dim_equipment = distinct Silver equipment + Unknown row",
      silver_equipment_count + 1, spark.table(DIM_EQUIPMENT).count())

assert not FAILED_CHECKS, f"Dimension checks failed: {FAILED_CHECKS}"

As-of timestamp: 2026-09-21 23:49:34
data2026acc123456.team3.test_gold_dim_date: 246 rows
data2026acc123456.team3.test_gold_dim_site: 3 rows
data2026acc123456.team3.test_gold_dim_equipment_type: 10 rows
data2026acc123456.team3.test_gold_dim_equipment: 9590 rows
data2026acc123456.team3.test_gold_dim_failure_mode: 13 rows
data2026acc123456.team3.test_gold_dim_maintenance_type: 5 rows
PASS | date_key is unique in data2026acc123456.team3.test_gold_dim_date | expected=246.00 actual=246.00
PASS | site_key is unique in data2026acc123456.team3.test_gold_dim_site | expected=3.00 actual=3.00
PASS | equipment_type_key is unique in data2026acc123456.team3.test_gold_dim_equipment_type | expected=10.00 actual=10.00
PASS | equipment_key is unique in data2026acc123456.team3.test_gold_dim_equipment | expected=9,590.00 actual=9,590.00
PASS | failure_mode_key is unique in data2026acc123456.team3.test_gold_dim_failure_mode | expected=13.00 actual=13.00
PASS | maintenance_type_key is unique in data2026acc1

In [0]:
# ==============================================================================
# GOLD - CELL 2 of 3: FACT tables
# Run after the dimensions cell (it uses its table names, settings and helpers).
#   fact_sensor_reading          one reading per equipment, sensor and timestamp
#   fact_maintenance_work_order  one work order
#   fact_downtime                one non-overlapping downtime interval per equipment
# ==============================================================================
FAILED_CHECKS.clear()

# ---------------------------------------------------------------- 1. Read Silver and dimensions
silver_sensor = spark.table(SILVER_SENSOR_EVENTS)
silver_wo = spark.table(SILVER_WORK_ORDERS)
silver_history = spark.table(SILVER_MAINTENANCE_HISTORY)

dim_equipment = spark.table(DIM_EQUIPMENT).select("equipment_id", "equipment_key", "site_key", "equipment_type_key")
dim_failure_mode = spark.table(DIM_FAILURE_MODE).select("failure_mode_code", "failure_mode_key")
dim_maintenance_type = spark.table(DIM_MAINTENANCE_TYPE).select(
    F.col("maintenance_type").alias("wo_type"), "maintenance_type_key"
)

EQUIPMENT_KEYS = ["equipment_key", "site_key", "equipment_type_key"]


def add_keys(df, dim, join_column, key_columns):
    """Look up the dimension keys. No match (or NULL) -> -1, the Unknown member."""
    return df.join(dim, join_column, "left").fillna(UNKNOWN_KEY, subset=key_columns)


as_of = F.lit(AS_OF).cast("timestamp")

# ---------------------------------------------------------------- 2. fact_sensor_reading
sensor = silver_sensor.withColumn("equipment_id", clean_code("equipment_id"))

# Silver removes exact duplicates only. Here we keep ONE row per (equipment, sensor, timestamp):
# the one with the highest record_version.
one_per_reading = Window.partitionBy(
    F.coalesce("equipment_id", F.lit("")), "sensor_id", "event_timestamp"
).orderBy(F.col("record_version").desc(), F.col("event_id"))
sensor = sensor.withColumn("rn", F.row_number().over(one_per_reading)).filter("rn = 1")

sensor = add_keys(sensor, dim_equipment, "equipment_id", EQUIPMENT_KEYS)

fact_sensor_reading = sensor.select(
    date_key(F.col("event_timestamp")).alias("date_key"),
    "equipment_key", "site_key", "equipment_type_key",
    F.col("event_timestamp").alias("event_ts"),
    "equipment_id", "sensor_id", "sensor_type", "event_id", "operating_state",
    F.col("reading_value_canonical").alias("reading_value"),   # already in SI units
    "canonical_unit", "alarm_state", "alarm_code",
    F.coalesce(F.col("alarm_state").isin(ALARM_STATES), F.lit(False)).alias("is_alarm"),
    F.col("_batch_id").alias("batch_id"),
    F.col("_source_file").alias("source_file"),
)

# ---------------------------------------------------------------- 3. fact_maintenance_work_order
# From the maintenance history we take the activity count, the temporary-fix flag and the activity parts cost.
# The history cost is used only to recover a missing work order parts cost (the work order cost is the source).
history = (
    silver_history.filter(F.col("work_order_id").isNotNull())
    .groupBy("work_order_id")
    .agg(
        F.count(F.lit(1)).alias("activity_count"),
        F.max((F.col("outcome") == TEMP_FIX).cast("int")).alias("temp_fix"),
        F.sum("parts_cost_eur").alias("activity_parts_cost_eur"),
    )
)

wo = (
    silver_wo
    .withColumn("equipment_id", clean_code("equipment_id"))
    .join(history, "work_order_id", "left")
)
wo = add_keys(wo, dim_equipment, "equipment_id", EQUIPMENT_KEYS)
wo = add_keys(wo, dim_failure_mode, "failure_mode_code", ["failure_mode_key"])
wo = add_keys(wo, dim_maintenance_type, "wo_type", ["maintenance_type_key"])

is_cancelled = F.coalesce(F.col("status").isin(CANCELLED_STATUSES), F.lit(False))

# Closed work order with no parts cost (Silver nulled a negative value): recover from activities.
needs_recovery = F.coalesce(
    (F.col("status") == "Closed")
    & F.col("parts_cost_eur").isNull()
    & F.col("activity_parts_cost_eur").isNotNull(),
    F.lit(False),
)
wo = (
    wo.withColumn("parts_cost_recovered", needs_recovery)
      .withColumn(
          "parts_cost_eur",
          F.when(F.col("parts_cost_recovered"), F.col("activity_parts_cost_eur"))
           .otherwise(F.col("parts_cost_eur"))
           .cast("decimal(12,2)"),
      )
)

is_open = ~F.col("status").isin(CLOSED_STATUSES)

fact_work_order = wo.select(
    "work_order_id",
    date_key(F.col("created_ts")).alias("created_date_key"),
    date_key(F.col("due_ts")).alias("due_date_key"),
    date_key(F.col("closed_ts")).alias("closed_date_key"),
    "equipment_key", "site_key", "equipment_type_key", "failure_mode_key", "maintenance_type_key",
    "equipment_id", "priority", "status", "root_cause_code",
    "created_ts", "due_ts", "actual_start_ts", "closed_ts",
    "labor_hours", "labor_cost_eur", "parts_cost_eur",
    (F.coalesce("labor_cost_eur", F.lit(0)) + F.coalesce("parts_cost_eur", F.lit(0))).alias("maintenance_cost_eur"),
    F.when(~is_cancelled, hours_between(F.col("created_ts"), F.col("closed_ts"))).alias("hours_to_close"),
    is_open.alias("is_open"),
    # overdue = still open and the due date has passed
    F.coalesce(is_open & (F.col("due_ts") < as_of), F.lit(False)).alias("is_overdue"),
    F.col("parts_cost_recovered"),
    is_cancelled.alias("is_cancelled"),
    F.coalesce((F.col("status") == "Closed") & (F.col("closed_ts") > F.col("due_ts")),
               F.lit(False)).alias("is_late_closed"),
    F.coalesce("activity_count", F.lit(0)).alias("activity_count"),
    (F.coalesce("temp_fix", F.lit(0)) == 1).alias("has_temporary_fix"),
    F.col("_batch_id").alias("batch_id"),
    F.col("_source_file").alias("source_file"),
)

# ---------------------------------------------------------------- 4. fact_downtime
# Source: the downtime columns of the work orders.
downtime = (
    silver_wo.filter(F.col("downtime_start_ts").isNotNull() & F.col("downtime_type").isNotNull())
    .withColumn("equipment_id", clean_code("equipment_id"))
    .withColumn("is_ongoing", F.col("downtime_end_ts").isNull())
    # downtime that has not ended yet is counted until the as-of timestamp
    .withColumn("end_ts", F.coalesce("downtime_end_ts", as_of))
    .filter(F.col("end_ts") >= F.col("downtime_start_ts"))
    # rows without equipment are never compared with each other
    .withColumn("asset", F.coalesce("equipment_id", F.concat(F.lit("UNKNOWN-"), F.col("work_order_id"))))
    .withColumn("failure_mode", F.coalesce("failure_mode_code", F.lit("NONE")))
    .withColumn("is_unplanned", F.col("downtime_type") == "unplanned")
)

# Overlap rule: the interval that starts first wins. A later interval of the same asset
# starts only when the earlier ones have ended (0 hours if it lies completely inside).
earlier_rows = (
    Window.partitionBy("asset")
    .orderBy("downtime_start_ts", "end_ts", "work_order_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)
downtime = downtime.withColumn("latest_earlier_end", F.max("end_ts").over(earlier_rows))
downtime = downtime.withColumn(
    "start_clipped", F.greatest("downtime_start_ts", "latest_earlier_end")  # greatest ignores NULL
)
downtime = downtime.withColumn("end_clipped", F.greatest("end_ts", "start_clipped"))

# Repeat failure: unplanned, and the same asset + failure mode had an unplanned failure
# that ended less than 30 days before this one started.
earlier_same_mode = (
    Window.partitionBy("asset", "failure_mode")
    .orderBy("downtime_start_ts", "end_ts", "work_order_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)
downtime = downtime.withColumn(
    "earlier_failure_end", F.max(F.when(F.col("is_unplanned"), F.col("end_ts"))).over(earlier_same_mode)
)
is_repeat = (
    F.col("is_unplanned")
    & (F.col("failure_mode") != "NONE")
    & F.col("earlier_failure_end").isNotNull()
    & (F.col("downtime_start_ts") <= F.col("earlier_failure_end") + F.expr(f"INTERVAL {REPEAT_DAYS} DAYS"))
)

downtime = add_keys(downtime, dim_equipment, "equipment_id", EQUIPMENT_KEYS)
downtime = add_keys(downtime, dim_failure_mode, "failure_mode_code", ["failure_mode_key"])
downtime = add_keys(downtime, dim_maintenance_type, "wo_type", ["maintenance_type_key"])

fact_downtime = downtime.select(
    "work_order_id",
    date_key(F.col("start_clipped")).alias("date_key"),
    "equipment_key", "site_key", "equipment_type_key", "failure_mode_key", "maintenance_type_key",
    "equipment_id", "downtime_type",
    F.col("start_clipped").alias("downtime_start_ts"),
    F.col("end_clipped").alias("downtime_end_ts"),
    hours_between(F.col("start_clipped"), F.col("end_clipped")).alias("downtime_hours"),
    F.col("downtime_start_ts").alias("source_start_ts"),   # as delivered, before clipping
    hours_between(F.col("downtime_start_ts"), F.col("end_ts")).alias("original_hours"),
    "is_ongoing",
    F.coalesce(is_repeat, F.lit(False)).alias("is_repeat_failure"),
    F.col("_batch_id").alias("batch_id"),
    F.col("_source_file").alias("source_file"),
)

# ---------------------------------------------------------------- 5. Save
save_table(fact_sensor_reading, FACT_SENSOR_READING)
save_table(fact_work_order, FACT_WORK_ORDER)
save_table(fact_downtime, FACT_DOWNTIME)

# Layout: the sensor fact is the only large table and is filtered by asset and date,
# so we co-locate the data on those columns. The other tables are small: no tuning needed.
spark.sql(f"OPTIMIZE {FACT_SENSOR_READING} ZORDER BY (equipment_key, date_key)")

# ---------------------------------------------------------------- 6. Checks (Silver vs Gold)
sensor_f = spark.table(FACT_SENSOR_READING)
wo_f = spark.table(FACT_WORK_ORDER)
dt_f = spark.table(FACT_DOWNTIME)

# row counts
rows_in_grain = (
    silver_sensor.select(F.coalesce(clean_code("equipment_id"), F.lit("")), "sensor_id", "event_timestamp")
    .distinct().count()
)
check("sensor rows = distinct (equipment, sensor, timestamp) in Silver", rows_in_grain, sensor_f.count())
check("work order rows", silver_wo.count(), wo_f.count())
check("work_order_id is unique", wo_f.count(), wo_f.select("work_order_id").distinct().count())

valid_downtime = silver_wo.filter(
    F.col("downtime_start_ts").isNotNull() & F.col("downtime_type").isNotNull()
    & (F.coalesce("downtime_end_ts", as_of) >= F.col("downtime_start_ts"))
)
check("downtime rows", valid_downtime.count(), dt_f.count())

# totals
silver_hours = valid_downtime.agg(
    F.sum(hours_between(F.col("downtime_start_ts"), F.coalesce("downtime_end_ts", as_of)))
).first()[0]
check("downtime hours before clipping", silver_hours, dt_f.agg(F.sum("original_hours")).first()[0], 0.000001)

silver_cost = silver_wo.agg(
    F.sum(F.coalesce("labor_cost_eur", F.lit(0)) + F.coalesce("parts_cost_eur", F.lit(0)))
).first()[0]
recovered = wo_f.filter("parts_cost_recovered").agg(F.sum("parts_cost_eur")).first()[0]
check("maintenance cost EUR (Silver + recovered)",
      float(silver_cost or 0) + float(recovered or 0),
      wo_f.agg(F.sum("maintenance_cost_eur")).first()[0], 0.01)

# no overlapping downtime for the same asset
earlier_in_gold = (
    Window.partitionBy("equipment_key")
    .orderBy("downtime_start_ts", "downtime_end_ts", "work_order_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)
overlaps = (
    dt_f.filter(F.col("equipment_key") != UNKNOWN_KEY)
    .withColumn("latest_earlier_end", F.max("downtime_end_ts").over(earlier_in_gold))
    .filter(F.col("downtime_start_ts") < F.col("latest_earlier_end"))
    .count()
)
check("overlapping downtime intervals", 0, overlaps)

# every key in a fact must exist in its dimension
for fact_table, key, dim_table, dim_key in [
    (FACT_SENSOR_READING, "date_key", DIM_DATE, "date_key"),
    (FACT_SENSOR_READING, "equipment_key", DIM_EQUIPMENT, "equipment_key"),
    (FACT_SENSOR_READING, "site_key", DIM_SITE, "site_key"),
    (FACT_SENSOR_READING, "equipment_type_key", DIM_EQUIPMENT_TYPE, "equipment_type_key"),
    (FACT_WORK_ORDER, "created_date_key", DIM_DATE, "date_key"),
    (FACT_WORK_ORDER, "due_date_key", DIM_DATE, "date_key"),
    (FACT_WORK_ORDER, "closed_date_key", DIM_DATE, "date_key"),
    (FACT_WORK_ORDER, "equipment_key", DIM_EQUIPMENT, "equipment_key"),
    (FACT_WORK_ORDER, "failure_mode_key", DIM_FAILURE_MODE, "failure_mode_key"),
    (FACT_WORK_ORDER, "maintenance_type_key", DIM_MAINTENANCE_TYPE, "maintenance_type_key"),
    (FACT_DOWNTIME, "date_key", DIM_DATE, "date_key"),
    (FACT_DOWNTIME, "equipment_key", DIM_EQUIPMENT, "equipment_key"),
    (FACT_DOWNTIME, "failure_mode_key", DIM_FAILURE_MODE, "failure_mode_key"),
    (FACT_DOWNTIME, "maintenance_type_key", DIM_MAINTENANCE_TYPE, "maintenance_type_key"),
]:
    fact, dim = spark.table(fact_table), spark.table(dim_table)
    orphans = fact.join(dim, fact[key] == dim[dim_key], "left_anti").count()
    check(f"keys of {key} found in dimension ({fact_table.split('.')[-1]})", 0, orphans)

# information for the demo (not pass / fail)
print("\nInformation:")
print("  sensor rows removed to keep the grain:", silver_sensor.count() - sensor_f.count())
print("  downtime hours removed by overlap clipping:",
      dt_f.agg(F.sum(F.col("original_hours") - F.col("downtime_hours"))).first()[0])
print("  sensor / work order / downtime rows with Unknown equipment:",
      sensor_f.filter(F.col("equipment_key") == UNKNOWN_KEY).count(),
      wo_f.filter(F.col("equipment_key") == UNKNOWN_KEY).count(),
      dt_f.filter(F.col("equipment_key") == UNKNOWN_KEY).count())
print("  parts cost recovered from activities:",
      wo_f.filter("parts_cost_recovered").count(), "work orders, EUR", recovered)

h_cost = (silver_history.filter("work_order_id IS NOT NULL")
          .groupBy("work_order_id").agg(F.sum("parts_cost_eur").alias("h_parts")))
gap = (silver_wo.filter("parts_cost_eur IS NOT NULL").join(h_cost, "work_order_id")
       .agg(F.count("*").alias("n"),
            F.sum((F.abs(F.col("parts_cost_eur") - F.col("h_parts")) >= 0.01).cast("int")).alias("differ"),
            F.sum("parts_cost_eur").alias("wo_eur"),
            F.sum("h_parts").alias("act_eur")).first())
print("  parts cost work order vs activities: compared", gap["n"], "differ", gap["differ"],
      "| work order EUR", gap["wo_eur"], "| activities EUR", gap["act_eur"])

QUARANTINE = f"{BASE}.test_silver_dq_quarantine"
print("\nOpen quarantine items:")
(spark.table(QUARANTINE).filter("resolution_status = 'OPEN'")
 .groupBy("entity", "rule_id", "quarantine_type").count()
 .orderBy("entity", "rule_id").show(50, truncate=False))

assert not FAILED_CHECKS, f"Fact checks failed: {FAILED_CHECKS}"

data2026acc123456.team3.test_gold_fact_sensor_reading: 9742 rows
data2026acc123456.team3.test_gold_fact_maintenance_work_order: 9584 rows
data2026acc123456.team3.test_gold_fact_downtime: 5397 rows
PASS | sensor rows = distinct (equipment, sensor, timestamp) in Silver | expected=9,742.00 actual=9,742.00
PASS | work order rows | expected=9,584.00 actual=9,584.00
PASS | work_order_id is unique | expected=9,584.00 actual=9,584.00
PASS | downtime rows | expected=5,397.00 actual=5,397.00
PASS | downtime hours before clipping | expected=26,804.47 actual=26,804.47
PASS | maintenance cost EUR (Silver + recovered) | expected=5,538,431.75 actual=5,538,431.75
PASS | overlapping downtime intervals | expected=0.00 actual=0.00
PASS | keys of date_key found in dimension (test_gold_fact_sensor_reading) | expected=0.00 actual=0.00
PASS | keys of equipment_key found in dimension (test_gold_fact_sensor_reading) | expected=0.00 actual=0.00
PASS | keys of site_key found in dimension (test_gold_fact_sensor_r

In [0]:
# ==============================================================================
# GOLD - CELL 3 of 3: KPIs
# Run after the facts cell.
# Step 1: table kpi_equipment_month = the ADDITIVE pieces of every KPI per equipment and month
#         (hours, counts, cost). Pieces can be added up to any level (site, month, fleet).
# Step 2: view vw_kpi_equipment_month = the KPI formulas on top of those pieces.
# ==============================================================================
FAILED_CHECKS.clear()

# Σημείωση: Υποθέτουμε ότι οι μεταβλητές KPI_TABLE, KPI_VIEW_V2, UNKNOWN_KEY κλπ. 
# έχουν ήδη οριστεί στα προηγούμενα κελιά (όπως το είχαμε τρέξει επιτυχώς).

# ---------------------------------------------------------------- 1. KPI formulas
# Εδώ ακριβώς βρίσκεται η λογική που έφτιαξες σε DAX (διαιρέσεις και NULLIF για αποφυγή divide-by-zero)
KPI_FORMULAS = {
    "availability": "(SUM(calendar_hours) - SUM(downtime_hours)) / NULLIF(SUM(calendar_hours), 0)",
    "unplanned_downtime_hours": "SUM(unplanned_downtime_hours)",
    "unplanned_failures": "SUM(unplanned_failure_count)",
    "mtbf_hours": "SUM(operating_uptime_hours) / NULLIF(SUM(unplanned_failure_count), 0)",
    "mttr_hours": "SUM(unplanned_downtime_hours) / NULLIF(SUM(unplanned_failure_count), 0)",
    "overdue_work_orders": "SUM(overdue_work_order_count)",
    "alarm_rate": "SUM(alarm_count) / NULLIF(SUM(asset_days), 0)",
    "alarm_warning_rate": "SUM(alarm_warning_count) / NULLIF(SUM(asset_days), 0)",
    "alarm_critical_rate": "SUM(alarm_critical_count) / NULLIF(SUM(asset_days), 0)",
    "maintenance_cost_eur": "SUM(maintenance_cost_eur)",
    "repeat_failure_rate": "SUM(repeat_failure_count) / NULLIF(SUM(unplanned_failure_count), 0)",
    "work_orders": "SUM(work_order_count)",
}

def kpi_query(group_columns):
    """SQL that calculates all KPIs, grouped by the given columns (empty list = whole fleet)."""
    kpis = ",\n    ".join(f"{formula} AS {name}" for name, formula in KPI_FORMULAS.items())
    if not group_columns:
        return f"SELECT\n    {kpis}\nFROM {KPI_TABLE}"
    groups = ", ".join(group_columns)
    return f"SELECT {groups},\n    {kpis}\nFROM {KPI_TABLE}\nGROUP BY {groups}"

# ---------------------------------------------------------------- 2. Read Gold tables
equipment = spark.table(DIM_EQUIPMENT).filter(F.col("equipment_key") != UNKNOWN_KEY)

# Δυναμική ανάκτηση πεδίων λειτουργίας (operating_hours) απευθείας από τον Silver αν λείπουν
if "operating_hours_per_day" not in equipment.columns:
    silver_eq = spark.table(SILVER_EQUIPMENT).select(
        F.upper(F.trim(F.col("equipment_id"))).alias("equipment_id"),
        "operating_hours_per_day",
        "operating_days_per_week"
    ).dropDuplicates(["equipment_id"])
    equipment = equipment.join(silver_eq, on="equipment_id", how="left")

# Υπολογισμός operating share (ποσοστό λειτουργίας βάσει ωραρίου)
equipment = equipment.withColumn(
    "operating_share",
    (F.coalesce(F.col("operating_hours_per_day").cast("double"), F.lit(24.0)) / 24.0) * 
    (F.coalesce(F.col("operating_days_per_week").cast("double"), F.lit(7.0)) / 7.0)
)

# Εξαίρεση ορφανών/άγνωστων μηχανημάτων από τα τελικά KPIs
sensor_f = spark.table(FACT_SENSOR_READING).filter(F.col("equipment_key") != UNKNOWN_KEY)
wo_f = spark.table(FACT_WORK_ORDER).filter(F.col("equipment_key") != UNKNOWN_KEY)
dt_f = spark.table(FACT_DOWNTIME).filter(F.col("equipment_key") != UNKNOWN_KEY)

as_of = F.lit(AS_OF).cast("timestamp")
data_start = F.lit(
    min(
        spark.table(FACT_SENSOR_READING).agg(F.date_format(F.min("event_ts"), TS_FORMAT)).first()[0],
        spark.table(FACT_WORK_ORDER).agg(F.date_format(F.min("created_ts"), TS_FORMAT)).first()[0],
    )
).cast("timestamp")

def month_of(timestamp_column):
    """First day of the month of a timestamp, as a date."""
    return F.date_trunc("month", timestamp_column).cast("date")

# ---------------------------------------------------------------- 3. Calendar hours per equipment and month
months = (
    spark.table(DIM_DATE).filter(F.col("month_start_date").isNotNull())
    .select("month_start_date").distinct()
    .filter(F.col("month_start_date") <= as_of.cast("date"))
    .filter(F.add_months("month_start_date", 1) > data_start.cast("date"))
)
month_start = F.col("month_start_date").cast("timestamp")
next_month_start = F.add_months("month_start_date", 1).cast("timestamp")
period_start = F.greatest(month_start, data_start, F.col("install_date").cast("timestamp"))
period_end = F.least(next_month_start, as_of, F.date_add("decommission_date", 1).cast("timestamp"))

calendar = (
    equipment.select("equipment_key", "site_key", "equipment_type_key", "install_date", "decommission_date", "operating_share")
    .crossJoin(months)
    .withColumn("calendar_hours", F.greatest(F.lit(0.0), hours_between(period_start, period_end)))
    .drop("install_date", "decommission_date")
)

# ---------------------------------------------------------------- 4. Downtime hours per month
downtime_by_month = (
    dt_f
    .withColumn(
        "month_start_date",
        F.explode(F.sequence(month_of(F.col("downtime_start_ts")), month_of(F.col("downtime_end_ts")), F.expr("INTERVAL 1 MONTH"))),
    )
    .withColumn(
        "hours_in_month",
        F.greatest(
            F.lit(0.0),
            hours_between(
                F.greatest(F.col("downtime_start_ts"), F.col("month_start_date").cast("timestamp")),
                F.least(F.col("downtime_end_ts"), F.add_months("month_start_date", 1).cast("timestamp")),
            ),
        ),
    )
    .groupBy("equipment_key", "month_start_date")
    .agg(
        F.sum("hours_in_month").alias("downtime_hours"),
        F.sum(F.when(F.col("downtime_type") == "unplanned", F.col("hours_in_month")).otherwise(0.0)).alias("unplanned_downtime_hours"),
    )
)

# ---------------------------------------------------------------- 5. Counts per month
failures_by_month = (
    dt_f.filter(F.col("downtime_type") == "unplanned")
    .groupBy("equipment_key", month_of(F.col("source_start_ts")).alias("month_start_date"))
    .agg(
        F.count(F.lit(1)).alias("unplanned_failure_count"),
        F.sum(F.col("is_repeat_failure").cast("int")).alias("repeat_failure_count"),
    )
)

alarms_by_month = (
    sensor_f
    .groupBy("equipment_key", month_of(F.col("event_ts")).alias("month_start_date"))
    .agg(
        F.sum(F.col("is_alarm").cast("int")).alias("alarm_count"),
        F.sum((F.col("alarm_state") == "warning").cast("int")).alias("alarm_warning_count"),
        F.sum((F.col("alarm_state") == "alarm").cast("int")).alias("alarm_critical_count"),
    )
)

created_by_month = (
    wo_f.groupBy("equipment_key", month_of(F.col("created_ts")).alias("month_start_date"))
    .agg(F.count(F.lit(1)).alias("work_order_count"))
)

overdue_by_month = (
    wo_f.filter(F.col("is_overdue"))
    .groupBy("equipment_key", month_of(F.col("due_ts")).alias("month_start_date"))
    .agg(F.count(F.lit(1)).alias("overdue_work_order_count"))
)

cost_by_month = (
    wo_f.groupBy("equipment_key", month_of(F.coalesce("closed_ts", "created_ts")).alias("month_start_date"))
    .agg(F.sum("maintenance_cost_eur").alias("maintenance_cost_eur"))
)

# ---------------------------------------------------------------- 6. Put the pieces together
pieces = [downtime_by_month, failures_by_month, alarms_by_month,
          created_by_month, overdue_by_month, cost_by_month]

event_months = pieces[0].select("equipment_key", "month_start_date")
for piece in pieces[1:]:
    event_months = event_months.unionByName(piece.select("equipment_key", "month_start_date"))
event_months = event_months.distinct().withColumn("has_events", F.lit(True))

kpi = (
    calendar.join(event_months, ["equipment_key", "month_start_date"], "left")
    .filter((F.col("calendar_hours") > 0) | F.col("has_events").isNotNull())
    .drop("has_events")
)
for piece in pieces:
    kpi = kpi.join(piece, ["equipment_key", "month_start_date"], "left")

kpi_equipment_month = kpi.select(
    "equipment_key", "site_key", "equipment_type_key",
    date_key(F.col("month_start_date")).alias("month_start_date_key"),
    "calendar_hours",
    (F.col("calendar_hours") / 24.0).alias("asset_days"),
    
    # ----------------------------------------------------------------
    # KEY LOGIC: Operating Share implementation for accurate MTBF
    (F.col("calendar_hours") * F.col("operating_share")).alias("operating_hours"),
    F.coalesce("downtime_hours", F.lit(0.0)).alias("downtime_hours"),
    (F.greatest(F.lit(0.0), F.col("calendar_hours") - F.coalesce("downtime_hours", F.lit(0.0))) * F.col("operating_share")).alias("operating_uptime_hours"),
    # ----------------------------------------------------------------
    
    F.coalesce("unplanned_downtime_hours", F.lit(0.0)).alias("unplanned_downtime_hours"),
    F.coalesce("unplanned_failure_count", F.lit(0)).alias("unplanned_failure_count"),
    F.coalesce("repeat_failure_count", F.lit(0)).alias("repeat_failure_count"),
    F.coalesce("alarm_count", F.lit(0)).alias("alarm_count"),
    F.coalesce("alarm_warning_count", F.lit(0)).alias("alarm_warning_count"),
    F.coalesce("alarm_critical_count", F.lit(0)).alias("alarm_critical_count"),
    F.coalesce("work_order_count", F.lit(0)).alias("work_order_count"),
    F.coalesce("overdue_work_order_count", F.lit(0)).alias("overdue_work_order_count"),
    F.coalesce("maintenance_cost_eur", F.lit(0)).alias("maintenance_cost_eur"),
    as_of.alias("as_of_ts"),
)

# ---------------------------------------------------------------- 7. Save table and view
kpi_equipment_month.withColumn("gold_loaded_ts", F.current_timestamp())\
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(KPI_TABLE)
print(f"{KPI_TABLE}: {spark.table(KPI_TABLE).count()} rows")

spark.sql(
    f"CREATE OR REPLACE VIEW {KPI_VIEW} AS\n"
    + kpi_query(["equipment_key", "site_key", "equipment_type_key", "month_start_date_key"])
)
print(f"{KPI_VIEW}: created")

# ---------------------------------------------------------------- 8. Checks (KPI pieces vs facts)
totals = spark.table(KPI_TABLE).agg(
    F.sum("downtime_hours"), F.sum("unplanned_downtime_hours"), F.sum("unplanned_failure_count"),
    F.sum("repeat_failure_count"), F.sum("alarm_count"), F.sum("work_order_count"),
    F.sum("overdue_work_order_count"), F.sum("maintenance_cost_eur"),
).first()

unplanned_dt = dt_f.filter(F.col("downtime_type") == "unplanned")
check("downtime hours", dt_f.agg(F.sum("downtime_hours")).first()[0], totals[0], 0.000001)
check("unplanned downtime hours", unplanned_dt.agg(F.sum("downtime_hours")).first()[0], totals[1], 0.000001)
check("unplanned failures", unplanned_dt.count(), totals[2])
check("repeat failures", dt_f.filter(F.col("is_repeat_failure")).count(), totals[3])
check("alarms", sensor_f.filter(F.col("is_alarm")).count(), totals[4])
check("work orders", wo_f.count(), totals[5])
check("overdue work orders", wo_f.filter(F.col("is_overdue")).count(), totals[6])
check("maintenance cost EUR", wo_f.agg(F.sum("maintenance_cost_eur")).first()[0], totals[7], 0.01)

# ---------------------------------------------------------------- 9. Show the KPIs
print("\nKPIs for the whole fleet:")
spark.sql(kpi_query([])).show(truncate=False)

print("KPIs by site:")
spark.sql(kpi_query(["site_key"])).join(
    spark.table(DIM_SITE).select("site_key", "site_code"), "site_key"
).drop("site_key").show(truncate=False)

assert not FAILED_CHECKS, f"KPI checks failed: {FAILED_CHECKS}"

data2026acc123456.team3.test_gold_kpi_equipment_month: 65249 rows
data2026acc123456.team3.test_gold_vw_kpi_equipment_month_v2: created
PASS | downtime hours | expected=26,054.54 actual=26,054.54
PASS | unplanned downtime hours | expected=9,217.98 actual=9,217.98
PASS | unplanned failures | expected=1,211.00 actual=1,211.00
PASS | repeat failures | expected=35.00 actual=35.00
PASS | alarms | expected=253.00 actual=253.00
PASS | work orders (not cancelled) | expected=9,251.00 actual=9,251.00
PASS | overdue work orders | expected=31.00 actual=31.00
PASS | maintenance cost EUR | expected=5,381,319.74 actual=5,381,319.74
PASS | operating_share within 0..1 | expected=0.00 actual=0.00

+------------+------------+-----------------+--------------+------+-----------+------------------+
|equipment_id|install_date|decommission_date|status        |months|work_orders|downtime_hours    |
+------------+------------+-----------------+--------------+------+-----------+------------------+
|EQ-11340    |2


# Knowledge Graph: Nodes and Edges Tables


In [ ]:
# Knowledge graph cells for .claude/reference/test_pipeline.ipynb
# Paste after the last Gold cell ("GOLD - CELL 3 of 3: KPIs"), in this order, and run in the same session
# (the code cell reuses BASE, DIM_*, FACT_*, UNKNOWN_KEY, F, Window, hours_between, save_table, check, FAILED_CHECKS).


# ===== CELL 1 (markdown) =====
# GOLD - KNOWLEDGE GRAPH
#
# Two extra Gold tables that describe the plant as a graph, built only from the Gold tables above (nothing is inferred):
#
# | Table | Contents |
# |---|---|
# | `test_gold_kg_nodes` | Site, EquipmentType, Equipment, Sensor, FailureMode, MaintenanceType, WorkOrder |
# | `test_gold_kg_edges` | LOCATED_AT, IS_OF_TYPE, HAS_SENSOR, PERFORMED_ON, HAS_FAILURE_MODE, HAS_MAINTENANCE_TYPE, REPEATED_FAILURE, FOLLOWED_BY, EXPERIENCED_FAILURE, TYPE_PRONE_TO |
#
# Unknown members (key -1) and the failure mode NONE (planned work) are left out. Node ids are business keys
# (`Equipment:EQ-10001`), not surrogate keys.


# ===== CELL 2 (code) =====
# ==============================================================================
# GOLD - CELL 4 (optional): KNOWLEDGE GRAPH
# Run after the KPI cell. Builds two tables from the Gold tables above:
#   kg_nodes = one row per node  (Site, EquipmentType, Equipment, Sensor, FailureMode, MaintenanceType, WorkOrder)
#   kg_edges = one row per directed edge (direct links from the keys + derived links)
# Nothing is inferred: every node and edge comes from a Gold row, so counts reconcile with Gold.
# Unknown members (key -1) and the failure mode NONE (planned work) are left out of the graph.
# ==============================================================================
from functools import reduce
from pyspark.sql import DataFrame

FAILED_CHECKS.clear()

KG_NODES = f"{BASE}.test_gold_kg_nodes"
KG_EDGES = f"{BASE}.test_gold_kg_edges"
NO_FAILURE = "NONE"

# ---------------------------------------------------------------- 1. Read Gold (known members only)
site = spark.table(DIM_SITE).filter(F.col("site_key") != UNKNOWN_KEY)
equipment_type = spark.table(DIM_EQUIPMENT_TYPE).filter(F.col("equipment_type_key") != UNKNOWN_KEY)
equipment = spark.table(DIM_EQUIPMENT).filter(F.col("equipment_key") != UNKNOWN_KEY)
failure_mode = spark.table(DIM_FAILURE_MODE).filter(
    (F.col("failure_mode_key") != UNKNOWN_KEY) & (F.col("failure_mode_code") != NO_FAILURE)
)
maintenance_type = spark.table(DIM_MAINTENANCE_TYPE).filter(F.col("maintenance_type_key") != UNKNOWN_KEY)

sensor_f = spark.table(FACT_SENSOR_READING).filter(
    (F.col("equipment_key") != UNKNOWN_KEY) & F.col("sensor_id").isNotNull()
)
wo_f = spark.table(FACT_WORK_ORDER)          # all work orders, also those without equipment
dt_f = spark.table(FACT_DOWNTIME)

eq_keys = equipment.select("equipment_key", "equipment_id", "equipment_type_key")


# ---------------------------------------------------------------- 2. Row builders
def node(df, node_type, key_column, label_column, properties, source):
    """One node row. node_id = '<type>:<business key>', so 'Equipment:EQ-10001' can be typed by a user or a tool.
    The extra attributes are packed into one JSON string (properties)."""
    return df.select(
        F.concat(F.lit(f"{node_type}:"), F.col(key_column)).alias("node_id"),
        F.lit(node_type).alias("node_type"),
        F.col(key_column).alias("business_key"),
        F.coalesce(F.col(label_column), F.col(key_column)).alias("label"),
        (F.to_json(F.struct(*properties)) if properties else F.lit(None).cast("string")).alias("properties"),
        F.lit(source).alias("source_table"),
    )


def edge(df, src_type, src_column, dst_type, dst_column, edge_type, source, weight=None, properties=None):
    """One edge row (src -> dst). weight is 1 for plain links, a count or a gap in days for derived links."""
    return df.select(
        F.concat(F.lit(f"{src_type}:"), F.col(src_column)).alias("src_id"),
        F.concat(F.lit(f"{dst_type}:"), F.col(dst_column)).alias("dst_id"),
        F.lit(edge_type).alias("edge_type"),
        (weight if weight is not None else F.lit(1.0)).cast("double").alias("weight"),
        (properties if properties is not None else F.lit(None).cast("string")).alias("properties"),
        F.lit(source).alias("source_table"),
    )


# ---------------------------------------------------------------- 3. Nodes
sensors = sensor_f.groupBy("sensor_id").agg(               # sensors become nodes, single readings do not
    F.max("sensor_type").alias("sensor_type"),
    F.count(F.lit(1)).alias("reading_count"),
    F.sum(F.col("is_alarm").cast("int")).alias("alarm_count"),
    F.max("event_ts").alias("last_reading_ts"),
)
downtime_by_wo = dt_f.groupBy("work_order_id").agg(
    F.max("downtime_type").alias("downtime_type"),
    F.sum("downtime_hours").alias("downtime_hours"),
)
work_orders = wo_f.join(downtime_by_wo, "work_order_id", "left")

kg_nodes = reduce(DataFrame.unionByName, [
    node(site, "Site", "site_code", "site_name",
         [F.col("country"), F.col("site_timezone").alias("timezone")], "test_gold_dim_site"),
    node(equipment_type, "EquipmentType", "equipment_type", "equipment_type", [], "test_gold_dim_equipment_type"),
    node(equipment, "Equipment", "equipment_id", "equipment_name",
         [F.col("manufacturer"), F.col("model_number"), F.col("production_line"),
          F.col("criticality"), F.col("status"), F.col("install_date")], "test_gold_dim_equipment"),
    node(failure_mode, "FailureMode", "failure_mode_code", "failure_mode_desc", [], "test_gold_dim_failure_mode"),
    node(maintenance_type, "MaintenanceType", "maintenance_type", "maintenance_type",
         [F.col("is_planned")], "test_gold_dim_maintenance_type"),
    node(sensors, "Sensor", "sensor_id", "sensor_id",
         [F.col("sensor_type"), F.col("reading_count"), F.col("alarm_count"), F.col("last_reading_ts")],
         "test_gold_fact_sensor_reading"),
    node(work_orders, "WorkOrder", "work_order_id", "work_order_id",
         [F.col("priority"), F.col("status"), F.col("root_cause_code"), F.col("created_ts"), F.col("closed_ts"),
          F.col("maintenance_cost_eur"), F.col("is_overdue"), F.col("has_temporary_fix"),
          F.col("downtime_type"), F.col("downtime_hours")], "test_gold_fact_maintenance_work_order"),
])

# ---------------------------------------------------------------- 4. Direct edges (they follow the Gold keys)
wo_links = wo_f.select("work_order_id", "equipment_key", "failure_mode_key", "maintenance_type_key")

located_at = edge(
    equipment.select("equipment_id", "site_key").join(site.select("site_key", "site_code"), "site_key"),
    "Equipment", "equipment_id", "Site", "site_code", "LOCATED_AT", "test_gold_dim_equipment")
is_of_type = edge(
    eq_keys.join(equipment_type.select("equipment_type_key", "equipment_type"), "equipment_type_key"),
    "Equipment", "equipment_id", "EquipmentType", "equipment_type", "IS_OF_TYPE", "test_gold_dim_equipment")
has_sensor = edge(
    sensor_f.select("equipment_key", "sensor_id").distinct()
    .join(eq_keys.select("equipment_key", "equipment_id"), "equipment_key"),
    "Equipment", "equipment_id", "Sensor", "sensor_id", "HAS_SENSOR", "test_gold_fact_sensor_reading")
performed_on = edge(                                        # work orders without equipment get no edge
    wo_links.join(eq_keys.select("equipment_key", "equipment_id"), "equipment_key"),
    "WorkOrder", "work_order_id", "Equipment", "equipment_id", "PERFORMED_ON", "test_gold_fact_maintenance_work_order")
has_failure_mode = edge(                                    # inner join: Unknown and NONE drop out
    wo_links.join(failure_mode.select("failure_mode_key", "failure_mode_code"), "failure_mode_key"),
    "WorkOrder", "work_order_id", "FailureMode", "failure_mode_code", "HAS_FAILURE_MODE",
    "test_gold_fact_maintenance_work_order")
has_maintenance_type = edge(
    wo_links.join(maintenance_type.select("maintenance_type_key", "maintenance_type"), "maintenance_type_key"),
    "WorkOrder", "work_order_id", "MaintenanceType", "maintenance_type", "HAS_MAINTENANCE_TYPE",
    "test_gold_fact_maintenance_work_order")

# ---------------------------------------------------------------- 5. Derived edges
# Basis: unplanned downtime of known equipment with a real failure mode (the same population as the KPI failures),
# put in order per equipment so each failure can see the one before it.
failures = (
    dt_f.filter((F.col("downtime_type") == "unplanned") & (F.col("equipment_key") != UNKNOWN_KEY))
    .join(failure_mode.select("failure_mode_key", "failure_mode_code"), "failure_mode_key")
    .select("work_order_id", "equipment_key", "failure_mode_code",
            "downtime_start_ts", "downtime_end_ts", "downtime_hours", "is_repeat_failure")
)
by_equipment = Window.partitionBy("equipment_key").orderBy("downtime_start_ts", "work_order_id")
by_equipment_mode = Window.partitionBy("equipment_key", "failure_mode_code").orderBy("downtime_start_ts", "work_order_id")
failures = (
    failures
    .withColumn("prev_wo", F.lag("work_order_id").over(by_equipment))
    .withColumn("prev_end", F.lag("downtime_end_ts").over(by_equipment))
    .withColumn("prev_mode", F.lag("failure_mode_code").over(by_equipment))
    .withColumn("prev_wo_same_mode", F.lag("work_order_id").over(by_equipment_mode))
    .withColumn("prev_end_same_mode", F.lag("downtime_end_ts").over(by_equipment_mode))
)

# REPEATED_FAILURE: a failure -> the earlier failure of the same asset and mode (flag from Gold, window = REPEAT_DAYS).
# weight = days between the end of the earlier failure and the start of this one.
repeated_failure = edge(
    failures.filter(F.col("is_repeat_failure") & F.col("prev_wo_same_mode").isNotNull()),
    "WorkOrder", "work_order_id", "WorkOrder", "prev_wo_same_mode", "REPEATED_FAILURE", "test_gold_fact_downtime",
    weight=F.round(hours_between(F.col("prev_end_same_mode"), F.col("downtime_start_ts")) / 24.0, 2),
    properties=F.to_json(F.struct(F.col("failure_mode_code").alias("failure_mode"))))

# FOLLOWED_BY: a failure -> the next unplanned failure of the same asset (any mode). weight = days between them.
followed_by = edge(
    failures.filter(F.col("prev_wo").isNotNull()),
    "WorkOrder", "prev_wo", "WorkOrder", "work_order_id", "FOLLOWED_BY", "test_gold_fact_downtime",
    weight=F.round(hours_between(F.col("prev_end"), F.col("downtime_start_ts")) / 24.0, 2),
    properties=F.to_json(F.struct(F.col("prev_mode").alias("from_mode"), F.col("failure_mode_code").alias("to_mode"))))

# EXPERIENCED_FAILURE: equipment -> failure mode, weight = number of failures
experienced = (
    failures.join(eq_keys.select("equipment_key", "equipment_id"), "equipment_key")
    .groupBy("equipment_id", "failure_mode_code")
    .agg(F.count(F.lit(1)).alias("failures"),
         F.round(F.sum("downtime_hours"), 2).alias("downtime_hours"),
         F.sum(F.col("is_repeat_failure").cast("int")).alias("repeat_failures"),
         F.max("downtime_start_ts").alias("last_failure_ts"))
)
experienced_failure = edge(
    experienced, "Equipment", "equipment_id", "FailureMode", "failure_mode_code",
    "EXPERIENCED_FAILURE", "test_gold_fact_downtime", weight=F.col("failures"),
    properties=F.to_json(F.struct("downtime_hours", "repeat_failures", "last_failure_ts")))

# TYPE_PRONE_TO: equipment type -> failure mode, weight = number of failures
prone = (
    failures.join(eq_keys, "equipment_key")
    .join(equipment_type.select("equipment_type_key", "equipment_type"), "equipment_type_key")
    .groupBy("equipment_type", "failure_mode_code")
    .agg(F.count(F.lit(1)).alias("failures"),
         F.countDistinct("equipment_id").alias("equipment_affected"),
         F.round(F.sum("downtime_hours"), 2).alias("downtime_hours"))
)
type_prone_to = edge(
    prone, "EquipmentType", "equipment_type", "FailureMode", "failure_mode_code",
    "TYPE_PRONE_TO", "test_gold_fact_downtime", weight=F.col("failures"),
    properties=F.to_json(F.struct("equipment_affected", "downtime_hours")))

kg_edges = reduce(DataFrame.unionByName, [
    located_at, is_of_type, has_sensor, performed_on, has_failure_mode, has_maintenance_type,
    repeated_failure, followed_by, experienced_failure, type_prone_to,
])

# ---------------------------------------------------------------- 6. Save
save_table(kg_nodes, KG_NODES)
save_table(kg_edges, KG_EDGES)
spark.sql(f"COMMENT ON TABLE {KG_NODES} IS 'Knowledge graph nodes derived from test_gold. node_id = <node_type>:<business_key>. Unknown members and failure mode NONE are excluded.'")
spark.sql(f"COMMENT ON TABLE {KG_EDGES} IS 'Knowledge graph edges derived from Gold (src_id -> dst_id). weight = 1, a count or a gap in days depending on edge_type.'")

# ---------------------------------------------------------------- 7. Checks (graph vs Gold)
nodes_df, edges_df = spark.table(KG_NODES), spark.table(KG_EDGES)


def nodes_of(node_type):
    return nodes_df.filter(F.col("node_type") == node_type).count()


def edges_of(edge_type):
    return edges_df.filter(F.col("edge_type") == edge_type).count()


check("node_id unique", 0, nodes_df.count() - nodes_df.select("node_id").distinct().count())
node_ids = nodes_df.select(F.col("node_id").alias("id"))
check("edges with an unknown source node", 0, edges_df.join(node_ids, edges_df.src_id == node_ids.id, "left_anti").count())
check("edges with an unknown target node", 0, edges_df.join(node_ids, edges_df.dst_id == node_ids.id, "left_anti").count())

check("Site nodes", site.count(), nodes_of("Site"))
check("EquipmentType nodes", equipment_type.count(), nodes_of("EquipmentType"))
check("Equipment nodes", equipment.count(), nodes_of("Equipment"))
check("FailureMode nodes", failure_mode.count(), nodes_of("FailureMode"))
check("MaintenanceType nodes", maintenance_type.count(), nodes_of("MaintenanceType"))
check("Sensor nodes", sensor_f.select("sensor_id").distinct().count(), nodes_of("Sensor"))
check("WorkOrder nodes", wo_f.count(), nodes_of("WorkOrder"))

failure_count = failures.count()
check("REPEATED_FAILURE edges", failures.filter(F.col("is_repeat_failure")).count(), edges_of("REPEATED_FAILURE"))
check("FOLLOWED_BY edges", failure_count - failures.select("equipment_key").distinct().count(), edges_of("FOLLOWED_BY"))
check("EXPERIENCED_FAILURE weight", failure_count,
      edges_df.filter(F.col("edge_type") == "EXPERIENCED_FAILURE").agg(F.sum("weight")).first()[0])
check("TYPE_PRONE_TO weight", failure_count,
      edges_df.filter(F.col("edge_type") == "TYPE_PRONE_TO").agg(F.sum("weight")).first()[0])

# ---------------------------------------------------------------- 8. Show the graph
print("\nNodes by type:")
nodes_df.groupBy("node_type").count().orderBy("node_type").show(truncate=False)
print("Edges by type:")
edges_df.groupBy("edge_type").count().orderBy("edge_type").show(truncate=False)

assert not FAILED_CHECKS, f"Knowledge graph checks failed: {FAILED_CHECKS}"
